# Illinois Cost Containment Report - Commercial Auto Liability (Line 19.4)

Independent notebook that builds the 17-column Illinois CSV (Part 4203, Subpart A).

**Independence guarantee.** This notebook never reads or writes any table or file used by the
STAT NISS notebook.

- Every table it creates starts with `TABLE_PREFIX` (default `IL_`).
- Its output goes to its own folder (`OUTPUT_FOLDER`).
- It reads the shared source tables, the shared lookup CSV files and the class workbook read-only.
- The STAT table `Final_Transformed_Data` is used only for an optional read-only reconciliation.

**What it does**

1. Rebuilds the policy side as the STAT notebook does (base query, duplicate removal, coverage
   merge, prorata, offset and onset records). The base query also reads the prior booking year, so
   premium booked last year can be earned this year.
2. Rebuilds the claim side (paid loss, reserve at December of the experience year, ALAE, counts)
   and links claims to policies. Only ASLOB 194 claims are read.
3. Applies the STAT attribute rules that decide ASLOB, class code and exposure.
4. Adds the policy dates and calculates earned premium by days.
5. Removes ASLOB 193 and 212 rows, then maps every vehicle class to 2A, 1A, 3C or OTHR with the
   class workbook. A class that is not on the Primary Class or Public Auto sheet becomes OTHR.
6. Saves the pre-report detail data, sums by State ID and class for Illinois-only (12) and
   Multi-state (MS), applies the Illinois rules and writes one clean 17-column CSV.

**NISS-only steps are not included:** ZIP and territory, exception codes, NISS coverage and
subline codes, fixed-width padding, Transmittal and Summary files.

**Before the first run**

1. Upload `Commercial_Auto_Classes.xlsx` to the Lakehouse Files folder and set `CLASS_WORKBOOK_PATH`.
2. Check the parameters in section 2. Rows marked pending in the table below still need the client.
3. Run the notebook. Read the log: row exclusions, class profile, STAT reconciliation and output checks.


## Decisions recorded in the defaults

| Item | Parameter | Value | Source |
|------|-----------|-------|--------|
| FEIN | `FEIN` | 230636660 | Last year's filed file |
| Filing method | `FILING_METHOD` | 6 | Development spec |
| Line of business | `LINE_OF_BUSINESS` | 194 | Development spec |
| Form type | `FORM_TYPE` | C | Development spec |
| Experience year | `REPORTING_YEAR` | 2026 (reports 2025) | Development spec |
| Outstanding ALAE | `OUTSTANDING_ALAE_MODE`, `COMBINED_LITERAL` | combined, COMBINED | Development spec |
| Personal auto classes | `INCLUDE_PERSONAL_AUTO_CLASSES` | True | instructions.txt |
| Claims | `LIABILITY_ASLOB_CODES` | 194 only, no 2026 correction, no property query | instructions.txt |
| Policy term filter | `POLICY_TERM_EFFECTIVE_AFTER` | empty (removed) | instructions.txt |
| Claim-side term and negative-reserve filters | `APPLY_CLAIM_TERM_FILTER`, `REMOVE_NEGATIVE_RESERVES` | False | instructions.txt, developer notebook |
| Reserve date | `RESERVE_MONTH` | December of the experience year | instructions.txt, Illinois instructions |
| ASLOB cleanup | `ASLOB_FILTER_MODE`, `ASLOB_EXCLUDE_CODES` | remove 193 and 212 | instructions.txt |
| Rows with no book date | `INCLUDE_NULL_BOOK_DATE` | True | Developer notebook |
| Earned premium | `EARNED_PREMIUM_LOOKBACK_YEARS`, two date column parameters | days based, 1 prior year | Developer notebook |
| Class mapping | `CLASS_WORKBOOK_PATH`, `CLASS_WORKBOOK_SHEETS` | workbook, others OTHR | Commercial_Auto_Classes.xlsx |
| Exposure on OTHR | `BLANK_EXPOSURE_FOR_OTHR` | True (blank) | Illinois instructions |
| One-off policy fixes | four fix parameters | empty | Decision |
| Header row | `WRITE_HEADER` | True for review, False for the final file | Decision |

## Pending client answers (a switch exists for each)

| Item | Parameter | Default now |
|------|-----------|-------------|
| Claim count basis | `CLAIM_COUNT_MODE` | `stat` (sum of STAT row counts). `per_claim` counts each claim once per class. |
| Classes whose exposure is miles or receipts (1B, 1C, 3B, 3C) | `MILES_RECEIPTS_CLASS_MODE` | `stop`. The run stops if such a class has data. `car_months` continues. |
| Earned premium method, State Page figures | `STATE_PAGE_WRITTEN_PREMIUM` | not set |


## 1. Imports


In [ ]:
import logging
from functools import reduce
from typing import Dict, List, Optional, Tuple

import pandas as pd
from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql import functions as F


## 2. Parameters

This is the Fabric parameters cell. Every value can be overridden from a pipeline.


In [ ]:
# ---- Company and filing identifiers ---------------------------------------
FEIN = "230636660"         # 9 digits, no hyphen (value from last year's filed file).
FILING_METHOD = "6"        # 1, 2, 3, 4, 6 or 7 (Section 4203.70). 6 = Company Direct (dev spec).
LINE_OF_BUSINESS = "194"   # Value from the development spec.
FORM_TYPE = "C"            # C = claims-made (dev spec), O = occurrence, T = claims-made tail.

# ---- Reporting period -----------------------------------------------------
REPORTING_YEAR = "2026"    # Data call year. The experience year is this value minus one.
STATISTICAL_YEAR = ""      # Optional override of the experience year, for example "2025".
EARNED_PREMIUM_LOOKBACK_YEARS = 1  # Prior booking years read so their premium can earn this year.

# ---- Shared sources (read-only) -------------------------------------------
POLICY_SCHEMA = "ODS_STATS_TBLS_LH"
CLAIMS_MONTHLY_TABLE = "DMAnalytics.FactClaimsMonthlyDataStore"
CLAIMS_MASTER_TABLE = "DMAnalytics.FactClaims"
CLAIM_COMPANY_CODE = "14044"
POLICY_LOB_CODE = "AC"
LOOKUP_FOLDER = "Files"
LOOKUP_FILES = {
    "duplicate_coverages": "DeleteDuplicateCoveragesFromSTATExtract.csv",
    "aslob_manual": "ASOLB_Manual_update.csv",
    "ba_plus_aslob": "BA PLUS-UpdateASLOB.csv",
    "has_pip": "HasPIPVUWC_New.csv",
    "niss_mapping": "NISS_Column_Mapping-Updated.csv",
    "class_corrections": "Corrected_Class_code.csv",
    "datafix_offsets": "DataFix-ForOffsetRecordsWithoutReferenceData.csv",
    "datafix_claims": "DataFix-Claims.csv",
}

# ---- Policy date columns used for earned premium (policy_policy_detail_info) ----
POLICY_TRANSACTION_EFFECTIVE_COLUMN = "TransactionEffectiveDate"
POLICY_TERM_EXPIRATION_COLUMN = "TermExpirationDate"

# ---- Own tables and outputs (never shared with the STAT notebook) ---------
TABLE_PREFIX = "IL_"
OUTPUT_FOLDER = "Files/IL_CostContainment"
OUTPUT_FILE_NAME = ""      # If empty, a name is built from line of business and year.
WRITE_HEADER = True        # True for review. Set False for the final Illinois upload file.
PRE_REPORT_FILE_NAME = "IL_PreReport_Data.csv"  # Row-level detail behind the report, for review.
WRITE_PRE_REPORT_CSV = True
STAT_RECONCILE_TABLE = "Final_Transformed_Data"  # Read-only check. Empty = skip.

# ---- Class mapping workbook (Commercial_Auto_Classes.xlsx) -----------------
CLASS_WORKBOOK_PATH = "/lakehouse/default/Files/Commercial_Auto_Classes.xlsx"  # Local path of the uploaded file.
CLASS_WORKBOOK_SHEETS = [          # Checked in this order. A class not found in any sheet becomes OTHR.
    {"sheet": "Public Auto", "header_row": 0, "code_column": "Primary Class",
     "class_column": "Cost Containment Class", "prefix_length": 4},
    {"sheet": "Primary Class", "header_row": 1, "code_column": "Primary Class",
     "class_column": "Cost Containment Class", "prefix_length": 3},
]
CLASS_SOURCE_COLUMN = "ClassCode"  # Vehicle class code column in the pipeline table.
BLANK_EXPOSURE_FOR_OTHR = True     # Instructions: exposure is not reported for OTHR.
MILES_RECEIPTS_CLASS_MODE = "stop"  # "stop" or "car_months" (pending client answer).
CLAIM_COUNT_MODE = "stat"          # "stat" or "per_claim" (pending client answer).

# ---- State identifiers in the output --------------------------------------
ILLINOIS_STATE_ID = "12"   # State Identifier for Illinois-only records.
MULTISTATE_ID = "MS"       # State Identifier for Multi-state records.

# ---- Scope rules (instructions.txt and the developer notebook) -------------
INCLUDE_PERSONAL_AUTO_CLASSES = True
PERSONAL_AUTO_CLASS_CODES = ["9410", "9437", "9582", "9235", "9670"]
INCLUDE_NULL_BOOK_DATE = True
POLICY_TERM_EFFECTIVE_AFTER = ""  # Empty = no policy term filter (instructions.txt). Or a date, for example "2023-06-30".
APPLY_CLAIM_TERM_FILTER = False
CLAIM_TERM_EFFECTIVE_FROM = "2023-07-01"
REMOVE_NEGATIVE_RESERVES = False
LIABILITY_ASLOB_CODES = ["194"]   # Claims query, and the rows kept when ASLOB_FILTER_MODE is keep_listed.
ASLOB_FILTER_MODE = "exclude_listed"   # "exclude_listed" (instructions.txt) or "keep_listed".
ASLOB_EXCLUDE_CODES = ["193", "212"]   # Removed when ASLOB_FILTER_MODE is exclude_listed.
EXCLUDED_COVERAGE_CODES = []      # Optional. For example ["BROAD PIP"] to treat it as no-fault.
BI_COVERAGE_CODES = ["BODINJLIAB", "BI"]  # Bodily injury rows carry the car-month exposure.
RESERVE_MONTH = ""                # Empty = December of the experience year (YYYY-12-01).

# ---- One-off fixes. Empty by default; fill only if the STAT owner confirms ----
DELETE_POLICY_PAIRS = []          # List of (PolicyNumber, PolicyVersion) to remove.
OFFSET_TARGET_FIX = ()            # (PolicyNumber, CoverageCode, ClaimCode) to keep as child records only.
CLAIM_DERIVED_NAME_FIX = ()       # (PolicyNumber, TransactionNumber, ClaimNumber, from name, to name).
CLAIM_COVERAGE_CODE_FIX = ()      # (ClaimNumber, ClaimCoverageName, new CoverageCode).

# ---- ALAE ------------------------------------------------------------------
OUTSTANDING_ALAE_MODE = "combined"  # "combined", "combined_when_outstanding" or "zero".
COMBINED_LITERAL = "COMBINED"       # Text for the Outstanding ALAE field (dev spec).

# ---- Optional final check against the Illinois State Page ------------------
STATE_PAGE_WRITTEN_PREMIUM = None   # Direct written premium, State Page line 19.4. None = skip.
STATE_PAGE_TOLERANCE_PERCENT = 5.0  # Allowed difference in percent.


## 3. Configuration and validation

Fixed values from the STAT notebook and the Illinois instructions, plus helpers that fail early
with a complete list of problems.


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("il_cost_containment")

# State codes stored by the pipeline (same values as the STAT notebook).
STATE_CODE_MAP: Dict[str, str] = {
    "DE": "07", "IL": "12", "IN": "13", "KS": "15",
    "OH": "34", "OK": "35", "PA": "37", "VA": "45",
}
ILLINOIS_ABBREVIATION = "IL"

# ---- STAT rule tables. Each table is copied from the STAT notebook; none is invented here. ----
SOURCE_IBMI = "IBMi"            # Policy source system names used by the STAT rules.
SOURCE_ORIGAMI = "Origami"
IBMI_POLICY_PREFIX = "GMCA"     # IBMi policy numbers start with this prefix.

# Question codes read from policy_question_answer.
QA_PRORATA_CODE = "ProrataPercentage"
QA_ITEM_NUMBER_CODE = "PrintUnitNumber"
QA_RATING_BASIS_CODE = "RatingBasis"

# Coverage extension rules (STAT, attribute rules).
IBMI_EXTENSION_BY_COVERAGE: Dict[str, str] = {
    "UPD DED": "UPD", "PIP LIMIT": "NF", "BI": "CBI", "BA PLUS": "CM", "BROAD PIP": "BI",
}
IBMI_EXTENSION_REPLACEMENTS: List[Tuple[List[str], str, str]] = [
    (["FARMTR", "FRMTRKPOLL", "HIRED AUTO", "NONOWNAUTO", "DOC"], "CBI", "BI"),
]
ORIGAMI_EXTENSION_BY_COVERAGE: Dict[str, str] = {"PIP LIMIT": "NF"}
IBMI_GROUP_BY_COVERAGE: Dict[str, str] = {"BA PLUS": "Property", "UPD DED": "Liability"}
BA_PLUS_COVERAGE = "BA PLUS"
BA_PLUS_EXTENSIONS_TO_CM = ["CBI", "BI"]
BA_PLUS_NEW_EXTENSION = "CM"
BA_PLUS_LISTED_ASLOB = 194
MED_EXP_COVERAGE = "MED EXP"
MED_EXP_DEFAULT_EXTENSION = "NF"
MED_EXP_PIP_NAME = "PIP LIMIT_MED EXP"
CLAIM_EXTENSIONS_TO_CM = ["RR", "TL"]

# PIP indicator rules: (coverage codes, rating basis values or None for any, indicator).
PIP_SOURCE_STATE = "PA"
PIP_EXTENSIONS = ["NF", "FPB"]
PIP_BASIS_RULES: List[Tuple[Tuple[str, ...], Optional[Tuple[int, ...]], int]] = [
    (("MED EXP", "INC LOSS", "ACC DEATH", "FUN EXP"), (3,), 1),
    (("MED EXP", "ACC DEATH", "INC LOSS", "FUN EXP"), (0, 1, 2, 4), 0),
    (("COMB-FPB",), (3,), 1),
    (("COMB-FPB",), (1, 2, 4), 0),
    (("EMB",), (1, 3), 1),
    (("EMB",), (2, 4), 0),
    (("BROAD PIP",), None, 0),
]

# NISS class code rules (STAT, class derivation).
OPTION1_VALUES = ["NI", "NI&R", "NS", "ST"]
OPTION1_DEFAULT_CLAIM_CODES = ["UBI__UBI", "UIM__UBI"]
OPTION1_DEFAULT_VALUE = "NS"
OPTION1_DEFAULT_STATE = "PA"
DEDUCTIBLE_CLAIM_CODES = ["OTC__CM", "COLLISION__CL"]
DEDUCTIBLE_STATE_CLAIM_CODE = "PIP LIMIT__NF"
DEDUCTIBLE_STATE = "DE"
IBMI_CLASS_OVERRIDES: Dict[str, int] = {
    "UBI HIRED": 9404, "UBI NONOWN": 9408, "UIM HIRED": 9404, "UIM NONOWN": 9408,
}
CLAIM_CODE_CLASS_OVERRIDES: Dict[str, int] = {"BA PLUS__CBI": 9999, "BA PLUS__BI": 9999}
VEHICLE_CLASS_MARKER = "Vehicle Class Code"

# Exposure rules (STAT, exposure).
EXPOSURE_DEFAULT_PRORATA = 1
MONTHS_PER_YEAR = 12
ALAE_MODES = ("combined", "combined_when_outstanding", "zero")

# Fixed values defined by the Illinois instructions.
OTHER_CLASS_CODE = "OTHR"
ILLINOIS_CLASS_CODES = ("1A", "1B", "1C", "2A", "3A", "3B", "3C", OTHER_CLASS_CODE)
VALID_FILING_METHODS = ("1", "2", "3", "4", "6", "7")
VALID_FORM_TYPES = ("C", "O", "T")
ROW_STATUS_KEPT = "KEPT"
ROW_KEPT_STATUSES = (ROW_STATUS_KEPT, "KEPT_ASLOB_MISSING")
EXPOSURE_BASE_OTHER = ("1B", "1C", "3B", "3C")  # Receipts or miles (Section 4203.50). The data has car-months only.
CLAIM_COUNT_MODES = ("stat", "per_claim")
ASLOB_FILTER_MODES = ("exclude_listed", "keep_listed")
EXPOSURE_CLASS_MODES = ("stop", "car_months")

# Coverage pairs whose amounts are merged into the primary coverage (STAT rule).
COVERAGE_MERGE_PAIRS: List[Tuple[str, str]] = [
    ("BODINJLIAB", "BI"), ("COLLISION", "COLL"), ("OTC", "COMP"),
]

# ASLOB assigned to IBMi policies by coverage code (STAT rule).
IBMI_ASLOB_BY_COVERAGE: Dict[str, int] = {
    "BA PLUS": 212, "BROAD PIP": 194, "BI": 194, "BLNKT AI": 194, "DOC": 194,
    "FARMTR": 194, "FRMTRKPOLL": 194, "GK COLL": 212, "GK COMP": 212,
    "HIRED AUTO": 194, "HIRED PHYS": 212, "NONOWNAUTO": 194, "PIP LIMIT": 193,
    "UBI HIRED": 194, "UBI NONOWN": 194, "UIM HIRED": 194, "UIM NONOWN": 194,
    "UPD DED": 194,
}

# ASLOB defaults for Origami policies when ASLOB is still empty (STAT rule).
ORIGAMI_DEFAULT_ASLOB: Dict[str, int] = {
    "BODINJLIAB": 194, "COLLISION": 212, "UIM": 194, "COMB-FPB": 193,
    "MED EXP": 193, "OTC": 212, "UBI": 194,
}

# Coverage and extension pairs that default HasPIPVUWCCoveredInd to 0 (STAT rule).
PIP_DEFAULT_PAIRS: List[str] = [
    "MED EXP_NF", "MED EXP_FPB", "COMB-FPB_NF", "COMB-FPB_FPB", "EMB_NF", "EMB_FPB",
    "BROAD PIP_NF", "BROAD PIP_FPB", "ACC DEATH_NF", "ACC DEATH_FPB",
    "FUN EXP_NF", "FUN EXP_FPB", "INC LOSS_NF", "INC LOSS_FPB",
]

# Claim coverage name and extension to derived coverage name (STAT rule).
CLAIM_COVERAGE_MAPPING: List[Tuple[str, str, str]] = [
    ("ACC DEATH", "NF", "ACC DEATH_NF"), ("BA PLUS", "CM", "BA PLUS_TE"),
    ("BA PLUS", "RR", "BA PLUS_RR"), ("BA PLUS", "TL", "BA PLUS_T-L"),
    ("BA PLUS_RR", "CM", "BA PLUS_RR"), ("BA PLUS_T-L", "CM", "BA PLUS_T-L"),
    ("BI", "BI", "BODINJLIAB"), ("BI", "CBI", "BODINJLIAB"),
    ("BODINJLIAB", "BI", "BODINJLIAB"), ("BODINJLIAB", "CBI", "BODINJLIAB"),
    ("COLL", "CL", "COLLISION"), ("COLLISION", "CL", "COLLISION"),
    ("COMB-FPB", "FPB", "COMB-FPB"), ("COMP", "CM", "OTC"),
    ("FRMTRKPOLL", "PD", "FRMTRKPOLL_PRPDMGLIAB"), ("GK COLL", "CL", "GK COLL"),
    ("GK COMP", "CM", "GK COMP"), ("PIP LIMIT_INC LOSS", "NF", "PIP LIMIT_INC LOSS"),
    ("PIP LIMIT_ACC DEATH", "NF", "PIP LIMIT_ACC DEATH"),
    ("PIP LIMIT_FUN EXP", "NF", "PIP LIMIT_FUN EXP"),
    ("HIRED AUTO", "PD", "HIRED AUTO_PRPDMGLIAB"), ("HIRED PHYS", "CM", "HIRED PHYS_OTC"),
    ("INC LOSS", "NF", "INC LOSS"), ("MED EXP", "FPB", "MED EXP"), ("MED EXP", "NF", "MED EXP"),
    ("MEDPAYMENT", "MP", "MEDPAYMENT"), ("MP", "MP", "MEDPAYMENT"), ("OTC", "CM", "OTC"),
    ("PD", "CPD", "PRPDMGLIAB"), ("PD", "PD", "PRPDMGLIAB"), ("PIP LIMIT", "NF", "PIP LIMIT"),
    ("PIP LIMIT_MED EXP", "NF", "PIP LIMIT_MED EXP"), ("PRPDMGLIAB", "CPD", "PRPDMGLIAB"),
    ("PRPDMGLIAB", "PD", "PRPDMGLIAB"), ("RR", "CM", "RR"), ("RR", "RR", "RR"),
    ("T&L", "TL", "T-L"), ("T-L", "CM", "T-L"), ("UBI", "UBI", "UBI"),
    ("UBI NS", "UBI", "UBI"), ("UIM NS", "UBI", "UIM"), ("UIM ST", "UBI", "UBI"),
    ("UPD", "UPD", "UPD"),
]

# Canonical measure names used from aggregation to output.
SUMMED_MEASURES: List[str] = [
    "written_premium", "earned_premium", "paid_loss", "outstanding_loss",
    "paid_alae", "written_exposure",
]
COUNT_MEASURES: List[str] = ["paid_claims", "outstanding_claims"]
MEASURES: List[str] = SUMMED_MEASURES + COUNT_MEASURES
AMOUNT_MEASURES: List[str] = [
    "written_premium", "earned_premium", "paid_loss", "outstanding_loss", "paid_alae",
]
ACTIVITY_MEASURES: List[str] = [
    "written_premium", "earned_premium", "paid_loss", "outstanding_loss",
]

# Exact header names and order of the 17 output columns (Section 4203.60).
OUTPUT_COLUMNS: List[str] = [
    "FEIN", "Filing Method", "Line of Business", "State ID", "Class Code",
    "Statistical Year", "Not Used", "Form Type", "Written Premium",
    "Earned Premium", "Paid Losses", "Outstanding Losses", "Paid ALAE",
    "Outstanding ALAE", "Written Exposures", "Number of Paid Claims",
    "Number of Outstanding Claims",
]


def resolve_statistical_year() -> str:
    """Return the experience year as a string.

    Returns:
        STATISTICAL_YEAR when provided, otherwise REPORTING_YEAR minus one.

    Raises:
        ValueError: If neither parameter is provided.
    """
    if STATISTICAL_YEAR:
        return str(STATISTICAL_YEAR)
    if REPORTING_YEAR:
        return str(int(REPORTING_YEAR) - 1)
    raise ValueError("Set STATISTICAL_YEAR or REPORTING_YEAR in the parameters cell.")


def year_window() -> Dict[str, object]:
    """Return the date boundaries derived from the statistical year.

    Returns:
        A dictionary with the year, the next year, the first booking year read, the first day
        of the year, the last payment month, the first day of the next year and the reserve month.
    """
    year = int(resolve_statistical_year())
    return {
        "year": year,
        "next_year": year + 1,
        "lookback_year": year - int(EARNED_PREMIUM_LOOKBACK_YEARS),
        "start": "%d-01-01" % year,
        "last_payment_month": "%d-12-01" % year,
        "end_of_year": "%d-01-01" % (year + 1),
        "reserve_month": RESERVE_MONTH or "%d-12-01" % year,
    }


def il_table(name: str) -> str:
    """Return a table name with the Illinois prefix so STAT tables are never touched.

    Args:
        name: Base table name.

    Returns:
        The prefixed table name.
    """
    return "%s%s" % (TABLE_PREFIX, name)


def lookup_path(key: str) -> str:
    """Return the path of a shared lookup CSV file.

    Args:
        key: Key in LOOKUP_FILES.

    Returns:
        The file path under LOOKUP_FOLDER.
    """
    return "%s/%s" % (LOOKUP_FOLDER, LOOKUP_FILES[key])


def quote_list(values) -> str:
    """Return values as a comma-separated list of SQL string literals.

    Args:
        values: Iterable of values.

    Returns:
        For example: 'a', 'b'.
    """
    return ", ".join("'%s'" % str(value).replace("'", "''") for value in values)


def validate_parameters() -> None:
    """Validate every parameter before any data is read.

    Raises:
        ValueError: With one message listing all invalid parameters.
    """
    errors = []
    if not (FEIN.isdigit() and len(FEIN) == 9):
        errors.append("FEIN must be exactly 9 digits with no hyphen.")
    if FILING_METHOD not in VALID_FILING_METHODS:
        errors.append("FILING_METHOD must be one of %s." % (VALID_FILING_METHODS,))
    if FORM_TYPE not in VALID_FORM_TYPES:
        errors.append("FORM_TYPE must be one of %s." % (VALID_FORM_TYPES,))
    if not (STATISTICAL_YEAR or REPORTING_YEAR):
        errors.append("Set STATISTICAL_YEAR or REPORTING_YEAR.")
    if int(EARNED_PREMIUM_LOOKBACK_YEARS) < 0:
        errors.append("EARNED_PREMIUM_LOOKBACK_YEARS must not be negative.")
    if not TABLE_PREFIX:
        errors.append("TABLE_PREFIX must not be empty so STAT tables are never overwritten.")
    if not OUTPUT_FOLDER or OUTPUT_FOLDER.rstrip("/") == LOOKUP_FOLDER:
        errors.append("OUTPUT_FOLDER must be its own sub-folder, not the shared lookup folder.")
    if OUTSTANDING_ALAE_MODE not in ALAE_MODES:
        errors.append("OUTSTANDING_ALAE_MODE must be one of %s." % (ALAE_MODES,))
    if CLAIM_COUNT_MODE not in CLAIM_COUNT_MODES:
        errors.append("CLAIM_COUNT_MODE must be one of %s." % (CLAIM_COUNT_MODES,))
    if ASLOB_FILTER_MODE not in ASLOB_FILTER_MODES:
        errors.append("ASLOB_FILTER_MODE must be one of %s." % (ASLOB_FILTER_MODES,))
    if MILES_RECEIPTS_CLASS_MODE not in EXPOSURE_CLASS_MODES:
        errors.append("MILES_RECEIPTS_CLASS_MODE must be one of %s." % (EXPOSURE_CLASS_MODES,))
    if ASLOB_FILTER_MODE == "exclude_listed" and not ASLOB_EXCLUDE_CODES:
        errors.append("ASLOB_EXCLUDE_CODES must not be empty when ASLOB_FILTER_MODE is exclude_listed.")
    if not LIABILITY_ASLOB_CODES:
        errors.append("LIABILITY_ASLOB_CODES must not be empty.")
    if not CLASS_WORKBOOK_PATH or not CLASS_WORKBOOK_SHEETS:
        errors.append("CLASS_WORKBOOK_PATH and CLASS_WORKBOOK_SHEETS must be set.")
    if not POLICY_TRANSACTION_EFFECTIVE_COLUMN or not POLICY_TERM_EXPIRATION_COLUMN:
        errors.append("The two policy date column names must be set.")
    if STATE_PAGE_TOLERANCE_PERCENT < 0:
        errors.append("STATE_PAGE_TOLERANCE_PERCENT must not be negative.")
    if errors:
        raise ValueError("Invalid parameters:\n - " + "\n - ".join(errors))


def validate_illinois_class_codes(codes: List[str]) -> None:
    """Validate that every class code is an Illinois Commercial Auto class code.

    Args:
        codes: Distinct Illinois class codes found in the class workbook.

    Raises:
        ValueError: If a code is not one of ILLINOIS_CLASS_CODES.
    """
    invalid = sorted(set(codes) - set(ILLINOIS_CLASS_CODES))
    if invalid:
        raise ValueError("Not valid Illinois class codes in the class workbook: %s" % invalid)


## 4. Utilities


In [ ]:
def get_spark() -> SparkSession:
    """Return the active Spark session provided by the Fabric runtime."""
    return SparkSession.builder.getOrCreate()


def read_lookup(spark: SparkSession, key: str) -> DataFrame:
    """Read a shared lookup CSV file (read-only).

    Args:
        spark: Active Spark session.
        key: Key in LOOKUP_FILES.

    Returns:
        The file as a DataFrame with a header and inferred types.
    """
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(lookup_path(key))
    )


def save_table(spark: SparkSession, df: DataFrame, table_name: str) -> DataFrame:
    """Save a DataFrame as a Delta table and return it re-read from storage.

    Re-reading cuts the long query plan, as the STAT notebook does with its checkpoint tables.

    Args:
        spark: Active Spark session.
        df: DataFrame to save.
        table_name: Target table name. Must start with TABLE_PREFIX.

    Returns:
        The saved table as a DataFrame.

    Raises:
        ValueError: If the table name does not start with TABLE_PREFIX.
    """
    if not table_name.startswith(TABLE_PREFIX):
        raise ValueError("Refusing to write %s: name must start with %s" % (table_name, TABLE_PREFIX))
    spark.sql("DROP TABLE IF EXISTS %s" % table_name)
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    logger.info("Saved table %s", table_name)
    return spark.table(table_name)


## 5. Policy side

Same logic as the STAT notebook: base query at four levels, duplicate removal, coverage merge,
Origami prorata and offset/onset reference records. Year, cut-off dates and the scope flags come
from the parameters.


In [ ]:
def base_policy_sql(window: Dict[str, object]) -> str:
    """Build the base policy query (risk, location, policy and child-premium levels).

    Args:
        window: Output of year_window.

    Returns:
        The Spark SQL text.
    """
    schema = POLICY_SCHEMA
    book_condition = "YEAR(PDI.BookDate) BETWEEN %d AND %d" % (window["lookback_year"], window["year"])
    if INCLUDE_NULL_BOOK_DATE:
        book_condition += " OR PDI.BookDate IS NULL"
    term_clause = ""
    if POLICY_TERM_EFFECTIVE_AFTER:
        term_clause = f"\n  AND PDI.TermEffectiveDate > '{POLICY_TERM_EFFECTIVE_AFTER}'"
    where_clause = f"""WHERE ({book_condition})
  AND ((CI.OnsetAmount IS NOT NULL AND CI.OnsetAmount <> 0)
    OR (CI.OffsetAmount IS NOT NULL AND CI.OffsetAmount <> 0)){term_clause}"""
    class_filter = ""
    if not INCLUDE_PERSONAL_AUTO_CLASSES:
        class_filter = "AND R.ClassCode NOT IN (%s)" % quote_list(PERSONAL_AUTO_CLASS_CODES)

    def select_list(risk_columns: str) -> str:
        """Return the common select list with the given risk columns."""
        return f"""PDI.PolicyNumber, PDI.PolicyVersion, PDI.InTransactionId, PDI.ReferencePolicyVersion,
    PDI.PolicySource, PDI.BookDate, PDI.ProrataPercent, PI.LOBCode,
    {risk_columns},
    PDI.PrimaryRiskState, CI.CoverageId, CI.ParentCoverageId, CI.RiskItemId, CI.ProductId,
    CI.CoverageCode, CI.CoverageExtension, CI.CoverageGroup, CI.OnsetAmount, CI.OffsetAmount,
    CI.Limit1, CI.Limit2, CI.ASLOB, CI.Option1, CI.Exposure, CI.Deductible1, CI.ClaimCode,
    CI.CoverageInfoGUID"""

    risk_columns = "R.RiskItemType, R.UnitNumber, R.ClassCode"
    null_risk_columns = (
        "CAST(NULL AS VARCHAR(50)) AS RiskItemType, CAST(NULL AS INT) AS UnitNumber, "
        "CAST(NULL AS VARCHAR(20)) AS ClassCode"
    )
    header = f"""FROM {schema}.policy_policy_detail_info PDI
JOIN {schema}.policy_Product_info PI
  ON PI.InTransactionId = PDI.InTransactionId
 AND PI.LOBCode = '{POLICY_LOB_CODE}'
 AND PI.ParentProductID IS NULL"""
    coverage_join = f"JOIN {schema}.policy_coverage_info CI"

    vehicle = f"""SELECT {select_list(risk_columns)}
{header}
JOIN {schema}.policy_risk_item_info R
  ON R.InTransactionId = PDI.InTransactionId AND R.RiskItemType = 'Vehicle' {class_filter}
{coverage_join}
  ON CI.InTransactionId = PDI.InTransactionId AND CI.RiskItemId = R.RiskItemId AND CI.ProductId IS NULL
{where_clause}"""
    location = f"""SELECT {select_list(risk_columns)}
{header}
JOIN {schema}.policy_risk_item_info R
  ON R.InTransactionId = PDI.InTransactionId AND R.RiskItemType = 'Location'
{coverage_join}
  ON CI.InTransactionId = PDI.InTransactionId AND CI.RiskItemId = R.RiskItemId AND CI.ProductId IS NULL
{where_clause}"""
    policy_level = f"""SELECT {select_list(null_risk_columns)}
{header}
{coverage_join}
  ON CI.InTransactionId = PDI.InTransactionId AND CI.ProductId = PI.ProductId AND CI.RiskItemId IS NULL
{where_clause}"""
    child_premium = f"""SELECT {select_list(null_risk_columns)}
{header}
{coverage_join}
  ON CI.InTransactionId = PDI.InTransactionId AND CI.ProductId IS NULL AND CI.RiskItemId IS NULL
{where_clause}"""
    return f"""SELECT DISTINCT *
FROM (
{vehicle}
UNION ALL
{location}
UNION ALL
{policy_level}
UNION ALL
{child_premium}
) U"""


def remove_known_deletions(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Remove policy versions listed in DELETE_POLICY_PAIRS and duplicate coverages.

    Args:
        spark: Active Spark session.
        df: Base policy rows.

    Returns:
        The rows without the listed policy versions and without the duplicate coverages
        listed in the shared duplicate coverage file.
    """
    if DELETE_POLICY_PAIRS:
        delete_df = spark.createDataFrame(DELETE_POLICY_PAIRS, ["PolicyNumber", "PolicyVersion"])
        df = df.join(delete_df, on=["PolicyNumber", "PolicyVersion"], how="left_anti")
    duplicates = read_lookup(spark, "duplicate_coverages")
    key_columns = ["PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageId"]
    return df.join(duplicates, on=key_columns, how="left_anti")


def merge_coverage_pairs(df: DataFrame) -> DataFrame:
    """Merge the amounts of secondary coverages into their primary coverage.

    For each pair in COVERAGE_MERGE_PAIRS, when a risk item and transaction hold both coverages,
    the secondary amounts are added to the primary row and the secondary row is removed.

    Args:
        df: Policy rows.

    Returns:
        The rows after merging.
    """
    result = df
    for primary, secondary in COVERAGE_MERGE_PAIRS:
        both = (
            result.groupBy("RiskItemId", "InTransactionId")
            .agg(
                F.sum(F.when(F.col("CoverageCode") == primary, 1).otherwise(0)).alias("p_cnt"),
                F.sum(F.when(F.col("CoverageCode") == secondary, 1).otherwise(0)).alias("s_cnt"),
                F.sum(F.when(F.col("CoverageCode") == secondary, F.col("OnsetAmount")).otherwise(0)).alias("sec_onset_sum"),
                F.sum(F.when(F.col("CoverageCode") == secondary, F.col("OffsetAmount")).otherwise(0)).alias("sec_offset_sum"),
            )
            .filter((F.col("p_cnt") > 0) & (F.col("s_cnt") > 0))
            .select("RiskItemId", "InTransactionId", "sec_onset_sum", "sec_offset_sum")
        )
        joined = result.join(both, ["RiskItemId", "InTransactionId"], "left")
        updated_primary = (
            joined.filter(F.col("CoverageCode") == primary)
            .withColumn(
                "OnsetAmount",
                F.when(F.col("sec_onset_sum").isNotNull(), F.col("OnsetAmount") + F.col("sec_onset_sum"))
                .otherwise(F.col("OnsetAmount")),
            )
            .withColumn(
                "OffsetAmount",
                F.when(F.col("sec_offset_sum").isNotNull(), F.col("OffsetAmount") + F.col("sec_offset_sum"))
                .otherwise(F.col("OffsetAmount")),
            )
            .drop("sec_onset_sum", "sec_offset_sum")
        )
        remaining = joined.filter(
            ~((F.col("CoverageCode") == secondary) & (F.col("sec_onset_sum").isNotNull()))
        ).drop("sec_onset_sum", "sec_offset_sum")
        result = remaining.filter(F.col("CoverageCode") != primary).unionByName(updated_primary)
    return result


def apply_origami_prorata(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Replace ProrataPercent for Origami policies with the value from the question table.

    Args:
        spark: Active Spark session.
        df: Policy rows.

    Returns:
        The rows with ProrataPercent updated for Origami policies.
    """
    answers = (
        spark.table("%s.policy_question_answer" % POLICY_SCHEMA)
        .filter(F.col("QuestionCode") == QA_PRORATA_CODE)
        .select(
            F.col("InTransactionId").alias("QA_InTransactionId"),
            F.col("TextValue").alias("Prorata_from_QA"),
        )
        .dropDuplicates(["QA_InTransactionId"])
    )
    joined = df.join(answers, df["InTransactionId"] == answers["QA_InTransactionId"], "left")
    updated = joined.withColumn(
        "ProrataPercent",
        F.when(F.col("PolicySource") == SOURCE_ORIGAMI, F.col("Prorata_from_QA")).otherwise(F.col("ProrataPercent")),
    )
    return updated.drop("QA_InTransactionId", "Prorata_from_QA")


In [ ]:
# Columns of the reference coverage joined to offset rows (used three times below).
REF_COVERAGE_COLUMNS = """c.CoverageId AS Ref_CoverageId,
    c.ParentCoverageId AS Ref_ParentCoverageId,
    c.CoverageExtension AS Ref_CoverageExtension,
    c.CoverageGroup AS Ref_CoverageGroup,
    c.Limit1 AS Ref_Limit1,
    c.Limit2 AS Ref_Limit2,
    c.ASLOB AS Ref_ASLOB,
    c.Option1 AS Option1_ref,
    c.Exposure AS Ref_Exposure,
    c.Deductible1 AS Ref_Deductible1,
    c.ClaimCode AS Ref_ClaimCode,
    c.CoverageInfoGUID AS Ref_CoverageInfoGUID"""


def build_offset_and_onset(spark: SparkSession, merged: DataFrame) -> DataFrame:
    """Attach reference attributes to offset rows and combine them with onset rows.

    An offset reverses earlier premium, so it takes its class, ASLOB and coverage attributes from
    the original (reference) transaction. Onset rows are kept as they are.

    Args:
        spark: Active Spark session.
        merged: Policy rows after merge and prorata.

    Returns:
        Onset rows plus offset rows enriched from their reference transaction.
    """
    schema = POLICY_SCHEMA

    def view(sql: str, name: str) -> None:
        """Run a query and register the result as a temporary view."""
        spark.sql(sql).createOrReplaceTempView(name)

    offsets = (
        merged.filter(F.col("OffsetAmount").isNotNull() & (F.col("OffsetAmount") != 0))
        .withColumn("OnsetAmount", F.lit(0))
        .withColumn("PolicyVersion_org", F.col("PolicyVersion"))
        .withColumn("InTransactionId_org", F.col("InTransactionId"))
        .withColumn("ReferencePolicyVersion_org", F.col("ReferencePolicyVersion"))
    )
    offsets.createOrReplaceTempView("il_df_offset")

    view(f"""SELECT o.*, pdi.InTransactionId AS Ref_InTransactionId,
    pdi.ReferencePolicyVersion AS Ref_ReferencePolicyVersion
FROM il_df_offset o
LEFT JOIN {schema}.policy_policy_detail_info pdi
  ON pdi.PolicyNumber = o.PolicyNumber AND pdi.PolicyVersion = o.ReferencePolicyVersion""", "il_ref_tx_df")
    view("SELECT * FROM il_ref_tx_df WHERE UnitNumber IS NOT NULL AND RiskItemId IS NOT NULL", "il_normal_df")
    view("SELECT * FROM il_ref_tx_df WHERE UnitNumber IS NULL OR RiskItemId IS NULL", "il_special_df")

    # Normal flow: risk-level offsets are matched to the reference risk item and coverage.
    view(f"""SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY InTransactionId, UnitNumber, RiskItemType
                                 ORDER BY RiskItemId) rn
    FROM {schema}.policy_risk_item_info) WHERE rn = 1""", "il_riskitem_filtered")
    view("""SELECT n.*, r.RiskItemId AS Ref_RiskItemId, r.ProductId AS Ref_ProductId,
    r.ClassCode AS Ref_ClassCode, r.RiskItemType AS Ref_RiskItemType
FROM il_normal_df n
LEFT JOIN il_riskitem_filtered r
  ON r.InTransactionId = n.Ref_InTransactionId AND r.UnitNumber = n.UnitNumber
 AND r.RiskItemType = n.RiskItemType""", "il_normal_risk_df")
    view(f"""SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY InTransactionId, RiskItemId, CoverageCode
                                 ORDER BY ParentCoverageId, CoverageId) rn
    FROM {schema}.policy_coverage_info) WHERE rn = 1""", "il_coverage_filtered_risk")
    view(f"""SELECT n.*, {REF_COVERAGE_COLUMNS}
FROM il_normal_risk_df n
LEFT JOIN il_coverage_filtered_risk c
  ON c.InTransactionId = n.Ref_InTransactionId AND c.RiskItemId = n.Ref_RiskItemId
 AND c.CoverageCode = n.CoverageCode""", "il_normal_joined")

    # Special flow: product-level offsets are matched to the reference product coverage.
    view(f"""SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY InTransactionId, ProductId, CoverageCode,
                                 CoverageExtension, ClaimCode ORDER BY ParentCoverageId, CoverageId) rn
    FROM {schema}.policy_coverage_info) WHERE rn = 1""", "il_coverage_filtered_prod")
    view(f"SELECT * FROM il_special_df WHERE PolicySource <> '{SOURCE_ORIGAMI}'", "il_special_non_origami")
    view(f"SELECT * FROM il_special_df WHERE PolicySource = '{SOURCE_ORIGAMI}'", "il_special_origami")
    non_origami = spark.sql(f"""SELECT s.*, NULL AS Ref_RiskItemId, NULL AS Ref_ClassCode,
    NULL AS Ref_RiskItemType, s.ProductId AS Ref_ProductId, {REF_COVERAGE_COLUMNS}
FROM il_special_non_origami s
LEFT JOIN il_coverage_filtered_prod c
  ON c.InTransactionId = s.Ref_InTransactionId AND c.ProductId = s.ProductId
 AND c.CoverageCode = s.CoverageCode""")
    origami = spark.sql(f"""SELECT so.*, NULL AS Ref_RiskItemId, NULL AS Ref_ClassCode,
    NULL AS Ref_RiskItemType, so.ProductId AS Ref_ProductId, {REF_COVERAGE_COLUMNS}
FROM il_special_origami so
LEFT JOIN il_coverage_filtered_prod c
  ON c.InTransactionId = so.Ref_InTransactionId AND c.ClaimCode = so.ClaimCode""")
    non_origami.unionByName(origami).createOrReplaceTempView("il_special_joined")

    final_columns = """PolicyNumber, PolicyVersion_org AS PolicyVersion,
    Ref_InTransactionId AS InTransactionId, ReferencePolicyVersion_org AS ReferencePolicyVersion,
    BookDate, ProrataPercent, PolicySource, LOBCode,
    {risk_type} AS RiskItemType, UnitNumber, {class_code} AS ClassCode, PrimaryRiskState,
    Ref_CoverageId AS CoverageId, Ref_ParentCoverageId AS ParentCoverageId,
    {risk_item} AS RiskItemId, {product} AS ProductId, CoverageCode,
    Ref_CoverageExtension AS CoverageExtension, Ref_CoverageGroup AS CoverageGroup,
    CAST(0 AS DECIMAL(18,2)) AS OnsetAmount, OffsetAmount, Ref_Limit1 AS Limit1,
    Ref_Limit2 AS Limit2, Ref_ASLOB AS ASLOB, Option1_ref AS Option1,
    Ref_Exposure AS Exposure, Ref_Deductible1 AS Deductible1, Ref_ClaimCode AS ClaimCode,
    Ref_CoverageInfoGUID AS CoverageInfoGUID"""
    normal_final = spark.sql(
        "SELECT " + final_columns.format(
            risk_type="Ref_RiskItemType", class_code="Ref_ClassCode",
            risk_item="Ref_RiskItemId", product="NULL")
        + " FROM il_normal_joined"
    )
    special_final = spark.sql(
        "SELECT " + final_columns.format(
            risk_type="RiskItemType", class_code="NULL",
            risk_item="NULL", product="Ref_ProductId")
        + " FROM il_special_joined"
    )
    reference = normal_final.unionByName(special_final)

    # One-off fix: for the target coverage keep only child records (as in the STAT notebook).
    if OFFSET_TARGET_FIX:
        policy_number, coverage_code, claim_code = OFFSET_TARGET_FIX
        is_target = (
            (F.col("PolicyNumber") == policy_number)
            & (F.col("CoverageCode") == coverage_code)
            & (F.col("ClaimCode") == claim_code)
        )
        reference = reference.filter(~is_target).unionByName(
            reference.filter(is_target).filter(F.col("ParentCoverageId").isNotNull())
        )

    onsets = (
        merged.filter(F.col("OnsetAmount").isNotNull() & (F.col("OnsetAmount") != 0))
        .withColumn("OffsetAmount", F.lit(0))
    )
    return onsets.unionByName(reference)


def build_policy_side(spark: SparkSession, window: Dict[str, object]) -> DataFrame:
    """Run the full policy side and save the result.

    Args:
        spark: Active Spark session.
        window: Output of year_window.

    Returns:
        The saved onset and offset table as a DataFrame.
    """
    base = spark.sql(base_policy_sql(window))
    cleaned = remove_known_deletions(spark, base)
    merged = apply_origami_prorata(spark, merge_coverage_pairs(cleaned))
    combined = build_offset_and_onset(spark, merged)
    return save_table(spark, combined, il_table("OnSet_And_Refrence_Offset"))


## 6. Claim side

Paid loss, reserve and ALAE come from the monthly claims table. The reserve is taken at
`RESERVE_MONTH` (December of the statistical year by default). Claims are then linked to the
policy coverage and risk item through the claim GUIDs, with the STAT fallback by item number.


In [ ]:
def coverage_mapping_sql() -> str:
    """Return the CoverageMapping query body built from CLAIM_COVERAGE_MAPPING."""
    rows = []
    for index, (name, extension, derived) in enumerate(CLAIM_COVERAGE_MAPPING):
        values = "'%s'" % name + ", '%s'" % extension + ", '%s'" % derived
        if index == 0:
            rows.append(
                "SELECT '%s' AS ClaimCoverageName, '%s' AS CoverageExtension, "
                "'%s' AS DerivedClaimCoverageName" % (name, extension, derived)
            )
        else:
            rows.append("UNION ALL SELECT %s" % values)
    return "\n".join(rows)


def liability_claims_sql(window: Dict[str, object]) -> str:
    """Build the liability claims query (payments, incurred and reserve by claim and coverage).

    Args:
        window: Output of year_window.

    Returns:
        The Spark SQL text.
    """
    codes = quote_list(LIABILITY_ASLOB_CODES)
    return f"""WITH CoverageMapping AS (
{coverage_mapping_sql()}
)
SELECT
    a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') AS ItemNumber,
    a.ClaimCoverageName, a.CoverageExtension, a.LossDate,
    YEAR(a.LossDate) AS AccidentYear,
    m.DerivedClaimCoverageName,
    CAST(0 AS DECIMAL(2)) AS PaidCount,
    CAST(0 AS DECIMAL(2)) AS ReserveCount,
    SUM(CASE WHEN a.BeginningOfMonth BETWEEN '{window["start"]}' AND '{window["last_payment_month"]}'
             THEN a.PaymentAmount ELSE 0 END) AS PaymentAmountDuration,
    SUM(CASE WHEN a.BeginningOfMonth < '{window["start"]}'
             THEN a.PaymentAmount ELSE 0 END) AS PaymentAmountDurationAsOfLastYear,
    SUM(CASE WHEN a.BeginningOfMonth < '{window["end_of_year"]}'
             THEN a.PaymentAmount ELSE 0 END) AS PaymentAmountDurationAsOfEndOfYear,
    SUM(CASE WHEN a.BeginningOfMonth < '{window["start"]}'
             THEN a.LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNOIBNRAsOfLastYear,
    SUM(CASE WHEN a.BeginningOfMonth < '{window["end_of_year"]}'
             THEN a.LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNOIBNRAsOfEndOfYear,
    SUM(CASE WHEN a.BeginningOfMonth = '{window["reserve_month"]}'
             THEN a.NetReserveAmount + a.ExcessDirectAmount ELSE 0 END) AS FinalReserveAmount
FROM {CLAIMS_MONTHLY_TABLE} a
LEFT JOIN CoverageMapping m
  ON a.ClaimCoverageName = m.ClaimCoverageName AND a.CoverageExtension = m.CoverageExtension
WHERE a.CompanyCode = '{CLAIM_COMPANY_CODE}'
  AND a.EASLobCode IN ({codes})
  AND YEAR(a.LossDate) < {window["next_year"]}
GROUP BY a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber, a.ItemNumber,
    a.ClaimCoverageName, a.CoverageExtension, m.DerivedClaimCoverageName, a.LossDate,
    a.PropertyOrLiability"""


def alae_claims_sql(window: Dict[str, object]) -> str:
    """Build the allocated loss adjustment expense query.

    The STAT notebook applies no company filter here. That behaviour is kept.

    Args:
        window: Output of year_window.

    Returns:
        The Spark SQL text.
    """
    codes = quote_list(LIABILITY_ASLOB_CODES)
    return f"""WITH CoverageMapping AS (
{coverage_mapping_sql()}
)
SELECT
    a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber, a.LossDate,
    YEAR(TO_DATE(a.LossDate, 'yyyyMMdd')) AS AccidentYear,
    a.PropertyOrLiability, a.ClaimCoverageName, a.CoverageExtension, a.LocationSequenceNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') AS ItemNumber,
    CAST(NULL AS DECIMAL(18,2)) AS PaidLoss,
    SUM(a.AllocatedLossAdjExp) AS AllocatedLossAdjExp,
    CAST(NULL AS DECIMAL(18,2)) AS ReserveAmount,
    m.DerivedClaimCoverageName
FROM {CLAIMS_MONTHLY_TABLE} a
LEFT JOIN CoverageMapping m
  ON a.ClaimCoverageName = m.ClaimCoverageName AND a.CoverageExtension = m.CoverageExtension
WHERE a.EASLobCode IN ({codes})
  AND a.BeginningOfMonth BETWEEN '{window["start"]}' AND '{window["last_payment_month"]}'
  AND a.AllocatedLossAdjExp <> 0
GROUP BY a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber,
    a.ClaimCoverageName, a.CoverageExtension, a.LocationSequenceNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', ''), m.DerivedClaimCoverageName, a.LossDate,
    a.PropertyOrLiability"""


# Columns shared by the paid, reserve and ALAE claim rows before they are combined.
CLAIM_BASE_COLUMNS: List[str] = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
    "ClaimCoverageName", "CoverageExtension", "LossDate", "AccidentYear",
    "DerivedClaimCoverageName", "PaidCount", "ReserveCount",
]


def build_claim_measures(spark: SparkSession, window: Dict[str, object]) -> DataFrame:
    """Build the paid loss, reserve and ALAE claim rows and combine them.

    Paid rows keep claims with a payment in the year. Reserve rows keep claims with a non-zero
    reserve. Counts follow the STAT rules: a paid claim counts 1 when all its payments fall in the
    year, -1 when a full prior payment is reversed. A reserve claim counts 1 when nothing has been
    paid and incurred is not zero.

    Args:
        spark: Active Spark session.
        window: Output of year_window.

    Returns:
        Paid, reserve and ALAE rows with the same columns.
    """
    liability = spark.sql(liability_claims_sql(window)).withColumn(
        "TransactionNumber", F.col("TransactionNumber").cast("int")
    )
    paid_to_date = F.col("PaymentAmountDurationAsOfEndOfYear")
    prior_paid = F.col("PaymentAmountDurationAsOfLastYear")
    in_year = F.col("PaymentAmountDuration")

    paid = (
        liability.filter(in_year != 0)
        .withColumn("PaidLossAmount", in_year)
        .withColumn(
            "PaidCount",
            F.when((in_year == paid_to_date) & (in_year > 0) & (paid_to_date > 0), 1)
            .when((in_year < 0) & (prior_paid > 0) & ((in_year + prior_paid) == 0), -1)
            .otherwise(0),
        )
        .select(
            *CLAIM_BASE_COLUMNS, "PaidLossAmount",
            F.lit(None).cast("double").alias("ReserveAmount"),
            F.lit(None).cast("double").alias("AllocatedLossAdjExp"),
        )
    )

    incurred_end = F.col("LossIncurredNOIBNRAsOfEndOfYear")
    incurred_prior = F.col("LossIncurredNOIBNRAsOfLastYear")
    reserve = (
        liability.filter(
            ~((paid_to_date == 0) & (incurred_end == 0) & (F.year("LossDate") == window["year"]))
        )
        .withColumn("ReserveAmount", F.col("FinalReserveAmount"))
        .filter(F.col("ReserveAmount") != 0)
        .withColumn(
            "ReserveCount",
            F.when((paid_to_date == 0) & (incurred_end != 0), 1)
            .when((paid_to_date == 0) & (incurred_end == 0) & (prior_paid == 0) & (incurred_prior != 0), -1)
            .otherwise(0),
        )
        .select(
            *CLAIM_BASE_COLUMNS, "ReserveAmount",
            F.lit(None).cast("double").alias("PaidLossAmount"),
            F.lit(None).cast("double").alias("AllocatedLossAdjExp"),
        )
    )

    alae = spark.sql(alae_claims_sql(window)).withColumn(
        "TransactionNumber", F.col("TransactionNumber").cast("int")
    )
    alae_rows = alae.select(
        "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
        "ClaimCoverageName", "CoverageExtension", "LossDate", "AccidentYear",
        "DerivedClaimCoverageName",
        F.lit(None).cast("integer").alias("PaidCount"),
        F.lit(None).cast("integer").alias("ReserveCount"),
        "AllocatedLossAdjExp",
        F.lit(None).cast("double").alias("ReserveAmount"),
        F.lit(None).cast("double").alias("PaidLossAmount"),
    )
    return reduce(lambda left, right: left.unionByName(right), [paid, reserve, alae_rows])


def attach_claim_policy_keys(spark: SparkSession, claims: DataFrame) -> DataFrame:
    """Attach the policy GUIDs and cause of loss to the claim rows.

    Args:
        spark: Active Spark session.
        claims: Output of build_claim_measures.

    Returns:
        The claim rows with ParentCoverageGUID, ChildCoverageGUID, CauseOfLoss and ParentRiskitemGUID.
    """
    guids = spark.sql(f"""SELECT PolicyNumber, TransactionNumber, ClaimNumber, ClaimantNumber, ItemNumber,
    ClaimCoverageName, CoverageExtension, CauseOfLoss, ParentCoverageGUID, ChildCoverageGUID,
    ParentRiskitemGUID
FROM {CLAIMS_MASTER_TABLE}
WHERE CompanyCode = '{CLAIM_COMPANY_CODE}' AND LOBCode = '{POLICY_LOB_CODE}'""")

    # Fill a missing TransactionNumber from the monthly table (one row per claim and claimant).
    monthly = spark.table(CLAIMS_MONTHLY_TABLE).dropDuplicates(["PolicyNumber", "ClaimNumber", "ClaimantNumber"])
    guids = (
        guids.alias("mg")
        .join(
            monthly.alias("fc"),
            on=[
                F.col("mg.PolicyNumber") == F.col("fc.PolicyNumber"),
                F.col("mg.ClaimNumber") == F.col("fc.ClaimNumber"),
                F.col("mg.ClaimantNumber") == F.col("fc.ClaimantNumber"),
            ],
            how="left",
        )
        .select(
            *[F.col("mg.%s" % name) for name in guids.columns if name != "TransactionNumber"],
            F.coalesce(F.col("mg.TransactionNumber"), F.col("fc.TransactionNumber")).alias("TransactionNumber"),
        )
    )

    # Normalise coverage extension and derived name (STAT rules).
    claims = claims.withColumn(
        "CoverageExtension",
        F.when(
            (F.col("ClaimCoverageName") == BA_PLUS_COVERAGE)
            & (~F.col("PolicyNumber").startswith(IBMI_POLICY_PREFIX)),
            BA_PLUS_NEW_EXTENSION,
        ).otherwise(F.col("CoverageExtension")),
    ).withColumn(
        "CoverageExtension",
        F.when(F.col("CoverageExtension").isin(CLAIM_EXTENSIONS_TO_CM), BA_PLUS_NEW_EXTENSION)
        .otherwise(F.col("CoverageExtension")),
    ).withColumn(
        "DerivedClaimCoverageName",
        F.when(
            (F.col("DerivedClaimCoverageName") == MED_EXP_COVERAGE)
            & (F.col("CoverageExtension") == MED_EXP_DEFAULT_EXTENSION)
            & (~F.col("PolicyNumber").startswith(IBMI_POLICY_PREFIX)),
            MED_EXP_PIP_NAME,
        ).otherwise(F.col("DerivedClaimCoverageName")),
    )

    mapping = guids.dropDuplicates([
        "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
        "ClaimCoverageName", "CoverageExtension", "ParentRiskitemGUID",
    ]).select(
        "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
        "ClaimCoverageName", "CoverageExtension", "ParentCoverageGUID", "ChildCoverageGUID",
        "CauseOfLoss", "ParentRiskitemGUID",
    )
    condition = (
        (F.col("f.PolicyNumber") == F.col("m.PolicyNumber"))
        & (F.col("f.ClaimNumber") == F.col("m.ClaimNumber"))
        & (F.col("f.ClaimantNumber") == F.col("m.ClaimantNumber"))
        & (F.col("f.ItemNumber").cast("int") == F.col("m.ItemNumber").cast("int"))
        & (F.col("f.DerivedClaimCoverageName") == F.col("m.ClaimCoverageName"))
        & (F.col("f.CoverageExtension") == F.col("m.CoverageExtension"))
        & (F.col("f.TransactionNumber").cast("int") == F.col("m.TransactionNumber").cast("int"))
    )
    return (
        claims.alias("f")
        .join(mapping.alias("m"), on=condition, how="left")
        .select("f.*", "m.ParentCoverageGUID", "m.ChildCoverageGUID", "m.CauseOfLoss", "m.ParentRiskitemGUID")
    )


In [ ]:
def filter_claims_for_scope(spark: SparkSession, claims: DataFrame) -> DataFrame:
    """Apply the claim-side scope filters controlled by the parameters.

    The term filter keeps claims whose policy term starts on or after CLAIM_TERM_EFFECTIVE_FROM.
    The reserve filter removes negative reserves and keeps rows that carry no reserve (paid loss
    and ALAE rows).

    Args:
        spark: Active Spark session.
        claims: Claim rows with policy keys.

    Returns:
        The filtered claim rows.
    """
    if APPLY_CLAIM_TERM_FILTER:
        policies = spark.table("%s.policy_policy_detail_info" % POLICY_SCHEMA)
        claims = (
            claims.alias("c")
            .join(
                policies.alias("p"),
                (F.col("c.PolicyNumber") == F.col("p.PolicyNumber"))
                & (F.col("c.TransactionNumber").cast("int") == F.col("p.PolicyVersion").cast("int")),
                "left",
            )
            .filter(F.col("p.TermEffectiveDate") >= F.lit(CLAIM_TERM_EFFECTIVE_FROM))
            .select("c.*")
        )
    if REMOVE_NEGATIVE_RESERVES:
        claims = claims.filter(F.col("ReserveAmount").isNull() | (F.col("ReserveAmount") >= 0))
    return claims


def enrich_claims_with_policy(spark: SparkSession, claims: DataFrame) -> DataFrame:
    """Join claims to the policy transaction, product, risk item and coverage through the GUIDs.

    Args:
        spark: Active Spark session.
        claims: Claim rows with policy keys.

    Returns:
        The claim rows with the matching policy attributes.
    """
    pdi = spark.table("%s.policy_policy_detail_info" % POLICY_SCHEMA).alias("pdi")
    product = spark.table("%s.policy_Product_info" % POLICY_SCHEMA).alias("pi")
    risk = spark.table("%s.policy_risk_item_info" % POLICY_SCHEMA).alias("ri")
    coverage = spark.table("%s.policy_coverage_info" % POLICY_SCHEMA).alias("ci")
    joined = (
        claims.alias("c")
        .join(
            pdi,
            (F.col("c.PolicyNumber") == F.col("pdi.PolicyNumber"))
            & (F.col("c.TransactionNumber").cast("int") == F.col("pdi.PolicyVersion").cast("int")),
            "left",
        )
        .join(product, F.col("pdi.InTransactionId") == F.col("pi.InTransactionId"), "left")
        .join(
            risk,
            (F.col("pdi.InTransactionId") == F.col("ri.InTransactionId"))
            & (F.upper(F.col("c.ParentRiskitemGUID")) == F.upper(F.col("ri.RiskItemInfoGUID"))),
            "left",
        )
        .join(
            coverage,
            (F.col("pdi.InTransactionId") == F.col("ci.InTransactionId"))
            & (F.upper(F.col("c.ParentCoverageGUID")) == F.upper(F.col("ci.CoverageInfoGUID"))),
            "left",
        )
    )
    return joined.select(
        F.col("c.*"),
        F.col("pdi.PolicyVersion"), F.col("pdi.InTransactionId"), F.col("pdi.ReferencePolicyVersion"),
        F.col("pdi.PolicySource"), F.col("pdi.BookDate"), F.col("pdi.PrimaryRiskState"),
        F.col("pdi.ProrataPercent"), F.col("pi.LOBCode"),
        F.col("ri.RiskItemType"), F.col("ri.UnitNumber"), F.col("ri.ClassCode"),
        F.col("ci.CoverageId"), F.col("ci.ParentCoverageId"), F.col("ci.RiskItemId"),
        F.col("ci.ProductId"), F.col("ci.CoverageCode"),
        F.col("ci.CoverageExtension").alias("PolicyCoverageExtension"),
        F.col("ci.CoverageGroup"), F.col("ci.OnsetAmount"), F.col("ci.OffsetAmount"),
        F.col("ci.Limit1"), F.col("ci.Limit2"), F.col("ci.ASLOB"), F.col("ci.Option1"),
        F.col("ci.Exposure"), F.col("ci.Deductible1"), F.col("ci.ClaimCode"),
        F.col("ci.CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
    )


def find_unmatched_claims(spark: SparkSession, claims: DataFrame) -> DataFrame:
    """Return claims whose coverage or risk item GUID is missing or not found in the policy tables.

    Args:
        spark: Active Spark session.
        claims: Claim rows with policy keys.

    Returns:
        The claim rows that could not be linked through the GUIDs.
    """
    claim_df = (
        claims.withColumn("ParentCoverageGUID", F.upper(F.trim(F.col("ParentCoverageGUID"))))
        .withColumn("ParentRiskitemGUID", F.upper(F.trim(F.col("ParentRiskitemGUID"))))
    )
    coverage_guids = (
        spark.table("%s.policy_coverage_info" % POLICY_SCHEMA)
        .select(F.upper(F.trim(F.col("CoverageInfoGUID"))).alias("PolicyCoverageGUID")).distinct()
    )
    risk_guids = (
        spark.table("%s.policy_risk_item_info" % POLICY_SCHEMA)
        .select(F.upper(F.trim(F.col("RiskItemInfoGUID"))).alias("PolicyRiskItemGUID")).distinct()
    )
    checked = (
        claim_df
        .join(coverage_guids, F.col("ParentCoverageGUID") == F.col("PolicyCoverageGUID"), "left")
        .join(risk_guids, F.col("ParentRiskitemGUID") == F.col("PolicyRiskItemGUID"), "left")
    )
    unmatched = checked.filter(
        F.col("ParentCoverageGUID").isNull() | F.col("ParentRiskitemGUID").isNull()
        | F.col("PolicyCoverageGUID").isNull() | F.col("PolicyRiskItemGUID").isNull()
    )
    if CLAIM_DERIVED_NAME_FIX:
        policy_number, transaction, claim_number, from_name, to_name = CLAIM_DERIVED_NAME_FIX
        unmatched = unmatched.withColumn(
            "DerivedClaimCoverageName",
            F.when(
                (F.col("PolicyNumber") == policy_number)
                & (F.col("TransactionNumber") == transaction)
                & (F.col("ClaimNumber") == claim_number)
                & (F.col("ClaimCoverageName") == from_name),
                to_name,
            ).otherwise(F.col("DerivedClaimCoverageName")),
        )
    return unmatched


def rebuild_unmatched_claims(spark: SparkSession, unmatched: DataFrame) -> DataFrame:
    """Link unmatched claims to the policy by item number, or by coverage when the item is zero.

    Args:
        spark: Active Spark session.
        unmatched: Output of find_unmatched_claims.

    Returns:
        The unmatched claims with policy attributes, in the same layout as the linked claims.
    """
    schema = POLICY_SCHEMA
    pdi = spark.table("%s.policy_policy_detail_info" % schema)
    risk = spark.table("%s.policy_risk_item_info" % schema)
    coverage = spark.table("%s.policy_coverage_info" % schema)
    answers = spark.table("%s.policy_question_answer" % schema)

    item_numbers = (
        answers.filter(F.col("QuestionCode") == QA_ITEM_NUMBER_CODE)
        .select("InTransactionId", "RiskItemId", F.col("TextValue").alias("ItemNumber"))
    )
    policy_df = (
        pdi.alias("pdi")
        .join(risk.alias("ri"), F.col("pdi.InTransactionId") == F.col("ri.InTransactionId"))
        .join(
            coverage.alias("ci"),
            (F.col("pdi.InTransactionId") == F.col("ci.InTransactionId"))
            & (F.col("ri.RiskItemId") == F.col("ci.RiskItemId")),
        )
        .join(
            item_numbers.alias("qa"),
            (F.col("pdi.InTransactionId") == F.col("qa.InTransactionId"))
            & (F.col("ri.RiskItemId") == F.col("qa.RiskItemId")),
        )
        .select(
            F.col("pdi.InTransactionId"), F.col("pdi.PolicyNumber"), F.col("pdi.PolicyVersion"),
            F.col("pdi.ReferencePolicyVersion"), F.col("pdi.PolicySource"), F.col("pdi.BookDate"),
            F.col("pdi.PrimaryRiskState"), F.col("pdi.ProrataPercent"), F.col("qa.ItemNumber"),
            F.col("ci.CoverageCode"), F.col("ci.CoverageExtension").alias("PolicyCoverageExtension"),
            F.col("ri.RiskItemType"), F.col("ri.UnitNumber"), F.col("ri.ClassCode"),
            F.col("ci.CoverageId"), F.col("ci.ParentCoverageId"), F.col("ci.RiskItemId"),
            F.col("ci.ProductId"), F.col("ci.CoverageGroup"), F.col("ci.OnsetAmount"),
            F.col("ci.OffsetAmount"), F.col("ci.Limit1"), F.col("ci.Limit2"), F.col("ci.ASLOB"),
            F.col("ci.Option1"), F.col("ci.Exposure"), F.col("ci.Deductible1"),
            F.col("ci.ClaimCode"), F.col("ci.CoverageInfoGUID"),
        )
    )
    window_spec = Window.partitionBy(
        "PolicyNumber", "PolicyVersion", "ItemNumber", "CoverageCode", "PolicyCoverageExtension"
    ).orderBy(F.col("CoverageId"))
    policy_dedup = (
        policy_df.withColumn("rn", F.row_number().over(window_spec)).filter(F.col("rn") == 1).drop("rn")
    )

    # Flow 1: claims with an item number are matched by policy, version, item, coverage and extension.
    by_item = (
        unmatched.filter(F.col("ItemNumber") != 0).alias("udf")
        .join(
            policy_dedup.alias("pol"),
            (F.col("udf.PolicyNumber") == F.col("pol.PolicyNumber"))
            & (F.col("udf.TransactionNumber").cast("int") == F.col("pol.PolicyVersion").cast("int"))
            & (F.col("udf.ItemNumber").cast("int") == F.col("pol.ItemNumber").cast("int"))
            & (F.col("udf.DerivedClaimCoverageName") == F.col("pol.CoverageCode"))
            & (F.col("udf.CoverageExtension") == F.col("pol.PolicyCoverageExtension")),
            "left",
        )
        .select(
            "udf.*",
            F.col("pol.InTransactionId"), F.col("pol.PolicyVersion"), F.col("pol.ReferencePolicyVersion"),
            F.col("pol.PolicySource"), F.col("pol.BookDate"), F.lit(POLICY_LOB_CODE).alias("LOBCode"),
            F.col("pol.PrimaryRiskState"), F.col("pol.ProrataPercent"), F.col("pol.RiskItemType"),
            F.col("pol.UnitNumber"), F.col("pol.CoverageCode"), F.col("pol.PolicyCoverageExtension"),
            F.col("pol.ClassCode"), F.col("pol.CoverageId"), F.col("pol.ParentCoverageId"),
            F.col("pol.RiskItemId"), F.col("pol.ProductId"), F.col("pol.CoverageGroup"),
            F.col("pol.OnsetAmount"), F.col("pol.OffsetAmount"), F.col("pol.Limit1"), F.col("pol.Limit2"),
            F.col("pol.ASLOB"), F.col("pol.Option1"), F.col("pol.Exposure"), F.col("pol.Deductible1"),
            F.col("pol.ClaimCode"), F.col("pol.CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
        )
    )

    # Flow 2: claims with item number zero are matched by transaction and coverage code only.
    policy_base = (
        unmatched.filter(F.col("ItemNumber") == 0).alias("udf")
        .join(
            pdi.alias("pdi"),
            (F.col("udf.PolicyNumber") == F.col("pdi.PolicyNumber"))
            & (F.col("udf.TransactionNumber") == F.col("pdi.PolicyVersion")),
            "left",
        )
        .select(
            "udf.*", F.col("pdi.InTransactionId"), F.col("pdi.PolicySource"), F.col("pdi.PolicyVersion"),
            F.col("pdi.ReferencePolicyVersion"), F.col("pdi.BookDate"), F.col("pdi.ProrataPercent"),
            F.col("pdi.PrimaryRiskState"),
        )
    )
    by_coverage = (
        policy_base.alias("base")
        .join(
            coverage.alias("ci"),
            (F.col("base.InTransactionId") == F.col("ci.InTransactionId"))
            & (F.col("base.ClaimCoverageName") == F.col("ci.CoverageCode")),
            "left",
        )
        .select(
            "base.*", F.lit(POLICY_LOB_CODE).alias("LOBCode"),
            F.lit(None).alias("RiskItemType"), F.lit(None).alias("UnitNumber"), F.lit(None).alias("ClassCode"),
            F.col("ci.CoverageId"), F.col("ci.ParentCoverageId"), F.col("ci.RiskItemId"), F.col("ci.ProductId"),
            F.col("ci.CoverageCode"), F.col("ci.CoverageExtension").alias("PolicyCoverageExtension"),
            F.col("ci.CoverageGroup"),
            F.lit(None).cast("double").alias("OnsetAmount"), F.lit(None).cast("double").alias("OffsetAmount"),
            F.col("ci.Limit1"), F.col("ci.Limit2"), F.col("ci.ASLOB"), F.col("ci.Option1"),
            F.col("ci.Exposure"), F.col("ci.Deductible1"), F.col("ci.ClaimCode"),
            F.col("ci.CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
        )
    )
    return by_item.unionByName(by_coverage).select(
        "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
        "ClaimCoverageName", "CoverageExtension", "LossDate", "AccidentYear", "DerivedClaimCoverageName",
        "PaidCount", "ReserveCount", "PaidLossAmount", "ReserveAmount", "AllocatedLossAdjExp",
        "ParentCoverageGUID", "ChildCoverageGUID", "CauseOfLoss", "ParentRiskitemGUID",
        "PolicyVersion", "InTransactionId", "ReferencePolicyVersion", "PolicySource", "BookDate",
        F.lit(None).alias("ProrataPercent"), "PrimaryRiskState", "LOBCode", "RiskItemType",
        "UnitNumber", "ClassCode", "CoverageId", "ParentCoverageId", "RiskItemId", "ProductId",
        "CoverageCode", "PolicyCoverageExtension", "CoverageGroup",
        F.lit(None).cast("double").alias("OnsetAmount"), F.lit(None).cast("double").alias("OffsetAmount"),
        "Limit1", "Limit2", "ASLOB", "Option1", F.lit(None).alias("Exposure"), "Deductible1",
        "ClaimCode", "PolicyCoverageInfoGUID",
    )


def build_claim_side(spark: SparkSession, window: Dict[str, object]) -> DataFrame:
    """Run the full claim side and save the result.

    Args:
        spark: Active Spark session.
        window: Output of year_window.

    Returns:
        The saved claim and policy table as a DataFrame.
    """
    claims = filter_claims_for_scope(spark, attach_claim_policy_keys(spark, build_claim_measures(spark, window)))
    enriched = enrich_claims_with_policy(spark, claims)
    unmatched = find_unmatched_claims(spark, claims)
    fallback = rebuild_unmatched_claims(spark, unmatched)

    # Replace the claim rows that could not be linked with their fallback version.
    keys = ["PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ClaimCoverageName", "CoverageExtension"]
    kept = enriched.join(fallback.select(*keys).distinct(), on=keys, how="leftanti")
    combined = (
        kept.unionByName(fallback)
        .withColumnRenamed("CoverageExtension", "ClaimCoverageExtension")
        .withColumnRenamed("PolicyCoverageInfoGUID", "CoverageInfoGUID")
        .withColumnRenamed("PolicyCoverageExtension", "CoverageExtension")
    )
    return save_table(spark, combined, il_table("Claim_Policy_NewApproachData"))


## 7. Attribute rules

The STAT rules that decide the extension, ASLOB, class code and exposure of every row. Rows without
a ClaimCode cannot be mapped to a NISS class and are not carried forward, exactly as in the STAT
notebook.


In [ ]:
def combine_policy_and_claims(spark: SparkSession) -> DataFrame:
    """Union the policy rows and the claim rows.

    Args:
        spark: Active Spark session.

    Returns:
        One DataFrame with premium rows (ClaimNumber is null) and claim rows.
    """
    policy = spark.table(il_table("OnSet_And_Refrence_Offset"))
    claims = (
        spark.table(il_table("Claim_Policy_NewApproachData"))
        .withColumn("OffsetAmount", F.lit(0))
        .withColumn("OnsetAmount", F.lit(0))
        .withColumn("Exposure", F.lit(None))
        .withColumn("ProrataPercent", F.lit(None))
    )
    return policy.unionByName(claims, allowMissingColumns=True)


def apply_extension_group_and_aslob(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Apply the coverage extension, coverage group and ASLOB rules (STAT rule tables).

    Args:
        spark: Active Spark session.
        df: Combined policy and claim rows.

    Returns:
        The rows with corrected CoverageExtension, CoverageGroup and ASLOB.
    """
    ibmi = F.col("PolicySource") == SOURCE_IBMI
    origami = F.col("PolicySource") == SOURCE_ORIGAMI
    coverage = F.col("CoverageCode")

    # Coverage extension. The coverage sets are disjoint, so the order of the rules does not matter.
    extension = F.col("CoverageExtension")
    for coverages, from_extension, to_extension in IBMI_EXTENSION_REPLACEMENTS:
        extension = F.when(
            ibmi & coverage.isin(coverages) & (F.col("CoverageExtension") == from_extension),
            F.lit(to_extension),
        ).otherwise(extension)
    for coverage_code, value in IBMI_EXTENSION_BY_COVERAGE.items():
        extension = F.when(ibmi & (coverage == coverage_code), F.lit(value)).otherwise(extension)
    for coverage_code, value in ORIGAMI_EXTENSION_BY_COVERAGE.items():
        extension = F.when(origami & (coverage == coverage_code), F.lit(value)).otherwise(extension)
    df = df.withColumn("CoverageExtension", extension)

    # Coverage group.
    group = F.col("CoverageGroup")
    for coverage_code, value in IBMI_GROUP_BY_COVERAGE.items():
        group = F.when(ibmi & (coverage == coverage_code), F.lit(value)).otherwise(group)
    df = df.withColumn("CoverageGroup", group)

    # ASLOB for IBMi policies by coverage code.
    aslob = F.col("ASLOB")
    for coverage_code, value in IBMI_ASLOB_BY_COVERAGE.items():
        aslob = F.when(ibmi & (coverage == coverage_code), F.lit(value)).otherwise(aslob)
    df = df.withColumn("ASLOB", aslob)

    # Fill empty ASLOB from the shared manual correction file.
    manual = read_lookup(spark, "aslob_manual").select(
        "PolicySource", "CoverageCode", "CoverageExtension", F.col("Correct ASLOB").alias("ASLOB_new")
    )
    df = (
        df.join(manual, on=["PolicySource", "CoverageCode", "CoverageExtension"], how="left")
        .withColumn("ASLOB", F.when(F.col("ASLOB").isNull(), F.col("ASLOB_new")).otherwise(F.col("ASLOB")))
        .drop("ASLOB_new")
    )

    # Fill empty ASLOB for Origami policies from the default table.
    default_aslob = F.col("ASLOB")
    for coverage_code, value in ORIGAMI_DEFAULT_ASLOB.items():
        default_aslob = F.when(
            F.col("ASLOB").isNull() & origami & (coverage == coverage_code), F.lit(value)
        ).otherwise(default_aslob)
    df = df.withColumn("ASLOB", default_aslob)

    # Policies listed in the shared BA PLUS file take the listed ASLOB on premium rows.
    ba_plus = (
        read_lookup(spark, "ba_plus_aslob")
        .select("PolicyNumber", "PolicyVersion", "CoverageCode").dropDuplicates()
        .withColumn("match_flag", F.lit(1))
    )
    return (
        df.join(ba_plus, on=["PolicyNumber", "PolicyVersion", "CoverageCode"], how="left")
        .withColumn(
            "ASLOB",
            F.when(
                (F.col("match_flag") == 1) & F.col("ClaimNumber").isNull(), F.lit(BA_PLUS_LISTED_ASLOB)
            ).otherwise(F.col("ASLOB")),
        )
        .drop("match_flag")
    )


def apply_pip_indicator_and_claim_codes(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Derive HasPIPVUWCCoveredInd, then fix extensions, ClaimCode and Option1.

    HasPIPVUWCCoveredInd is one of the keys used to find the NISS class code.

    Args:
        spark: Active Spark session.
        df: Rows after the ASLOB rules.

    Returns:
        The rows with HasPIPVUWCCoveredInd, ClaimCode and Option1 populated.
    """
    # Rating basis for the source system and state that use it (STAT rule).
    valid_keys = (
        df.filter(
            (F.col("PolicySource") == SOURCE_IBMI) & (F.col("PrimaryRiskState") == PIP_SOURCE_STATE)
            & F.col("CoverageExtension").isin(PIP_EXTENSIONS) & F.col("RiskItemId").isNotNull()
        )
        .select("InTransactionId", "RiskItemId").distinct()
    )
    rating_basis = (
        spark.table("%s.policy_question_answer" % POLICY_SCHEMA)
        .filter(F.col("QuestionCode") == QA_RATING_BASIS_CODE)
        .select(
            F.col("InTransactionID").alias("InTransactionId"),
            F.col("RiskItemID").alias("RiskItemId"),
            F.col("TextValue").cast("int").alias("RB_TextValue"),
        )
    )
    valid_rb = valid_keys.join(rating_basis, ["InTransactionId", "RiskItemId"], "inner")
    original_columns = df.columns
    df = df.join(valid_rb, ["InTransactionId", "RiskItemId"], "left").select(*original_columns, "RB_TextValue")

    # Indicator from the rule table. The first matching rule wins.
    coverage = F.col("CoverageCode")
    basis = F.col("RB_TextValue")
    pip_expression = F.lit(None)
    for coverages, bases, indicator in reversed(PIP_BASIS_RULES):
        condition = coverage.isin(list(coverages))
        if bases is not None:
            condition = condition & basis.isin(list(bases))
        pip_expression = F.when(condition, F.lit(indicator)).otherwise(pip_expression)
    df = df.withColumn("HasPIPVUWCCoveredInd", pip_expression).drop("RB_TextValue")

    # Override with the shared lookup file where it has a value.
    lookup = read_lookup(spark, "has_pip").select(
        "PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageExtension",
        F.col("HasPIPVUWCCoveredInd").alias("HasPIPVUWCCoveredInd_new"),
    )
    df = (
        df.join(lookup, on=["PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageExtension"], how="left")
        .withColumn("HasPIPVUWCCoveredInd", F.coalesce(F.col("HasPIPVUWCCoveredInd_new"), F.col("HasPIPVUWCCoveredInd")))
        .drop("HasPIPVUWCCoveredInd_new")
        .dropDuplicates()
    )

    df = df.withColumn(
        "CoverageExtension",
        F.when(
            (F.col("CoverageCode") == BA_PLUS_COVERAGE) & F.col("CoverageExtension").isin(BA_PLUS_EXTENSIONS_TO_CM),
            BA_PLUS_NEW_EXTENSION,
        )
        .when(
            (F.col("CoverageCode") == MED_EXP_COVERAGE)
            & (F.col("CoverageExtension").isNull() | (F.col("CoverageExtension") == "")),
            MED_EXP_DEFAULT_EXTENSION,
        )
        .otherwise(F.col("CoverageExtension")),
    ).withColumn(
        "ClaimCode",
        F.when(
            F.col("ParentCoverageId").isNull() & F.col("ClaimCode").isNull(),
            F.concat_ws("__", F.col("CoverageCode"), F.col("CoverageExtension")),
        ).otherwise(F.col("ClaimCode")),
    ).withColumn(
        "Option1",
        F.when(
            F.col("ClaimCode").isin(OPTION1_DEFAULT_CLAIM_CODES) & F.col("Option1").isNull()
            & (F.col("PrimaryRiskState") == OPTION1_DEFAULT_STATE),
            OPTION1_DEFAULT_VALUE,
        ).otherwise(F.col("Option1")),
    )
    pip_default = F.concat_ws("_", F.col("CoverageCode"), F.col("CoverageExtension")).isin(PIP_DEFAULT_PAIRS)
    return df.withColumn(
        "HasPIPVUWCCoveredInd",
        F.when(pip_default & F.col("HasPIPVUWCCoveredInd").isNull(), 0).otherwise(F.col("HasPIPVUWCCoveredInd")),
    )


In [ ]:
def derive_niss_class_code(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Derive NISS_Class_Codes by joining the shared NISS mapping file.

    Rows with a ClaimCode are matched on ClaimCode, state, Option1, deductible and the
    HasPIPVUWCCoveredInd flag. Fixed class codes are then applied from the override tables, the
    vehicle class is used where the mapping says so, and the four-character class is replaced by
    the corrected NISS code. Rows without a ClaimCode cannot be mapped and are not carried
    forward, exactly as in the STAT notebook.

    Args:
        spark: Active Spark session.
        df: Rows with HasPIPVUWCCoveredInd, ClaimCode and Option1.

    Returns:
        The rows that have a ClaimCode, with a NISS_Class_Codes column.
    """
    claim_code = F.upper(F.trim(F.col("ClaimCode")))
    state = F.upper(F.trim(F.col("PrimaryRiskState")))
    keeps_deductible = (
        claim_code.isin(DEDUCTIBLE_CLAIM_CODES)
        | ((claim_code == DEDUCTIBLE_STATE_CLAIM_CODE) & (state == DEDUCTIBLE_STATE))
    )
    prepared = (
        df.withColumn("Option1_clean", F.when(F.col("Option1").isin(OPTION1_VALUES), F.col("Option1")).otherwise(None))
        .withColumn("Deductible1", F.when(~keeps_deductible, None).otherwise(F.col("Deductible1")))
        .withColumn("Deductible1_clean", F.when(F.col("Deductible1") == 0.0, None).otherwise(F.col("Deductible1")))
    )

    mapping = read_lookup(spark, "niss_mapping").withColumn(
        "Deductible", F.when(F.col("Deductible") == 0, None).otherwise(F.col("Deductible"))
    ).withColumn(
        "EffectiveCoverageCode", F.coalesce(F.col("ODS Child Coverage Code"), F.col("Coverage Code"))
    )
    window_spec = Window.partitionBy(
        "EffectiveCoverageCode", "State", "Coverage Extension", "Deductible",
        "Deductible Applies to Option1", "Covered By Worker's Compensation", "ClaimCode",
    ).orderBy(F.lit(1))
    mapping = mapping.withColumn("rn", F.row_number().over(window_spec)).filter(F.col("rn") == 1).drop("rn")

    option_match = (
        (F.col("fp.Option1_clean") == F.col("mp.Deductible Applies to Option1"))
        | (F.col("fp.Option1_clean").isNull() & F.col("mp.Deductible Applies to Option1").isNull())
    )
    deductible_match = (
        (F.col("fp.Deductible1_clean") == F.col("mp.Deductible"))
        | (F.col("fp.Deductible1_clean").isNull() & F.col("mp.Deductible").isNull())
    )
    worker_match = (
        (F.col("fp.HasPIPVUWCCoveredInd").cast("int") == F.col("mp.Covered By Worker's Compensation").cast("int"))
        | (F.col("fp.HasPIPVUWCCoveredInd").isNull() & F.col("mp.Covered By Worker's Compensation").isNull())
    )
    mapped = (
        prepared.filter(F.col("ClaimCode").isNotNull()).alias("fp")
        .join(
            mapping.alias("mp"),
            (F.col("fp.ClaimCode") == F.col("mp.ClaimCode"))
            & (F.col("fp.PrimaryRiskState") == F.col("mp.State"))
            & option_match & deductible_match & worker_match,
            "left",
        )
        .select("fp.*", F.col("mp.NISS Class Codes (6)").alias("NISS_Class_Codes"))
    )

    # Fixed class codes: coverage-based for IBMi, then claim-code-based.
    ibmi = F.col("PolicySource") == SOURCE_IBMI
    class_code = F.col("NISS_Class_Codes")
    for coverage_code, value in IBMI_CLASS_OVERRIDES.items():
        class_code = F.when(ibmi & (F.col("CoverageCode") == coverage_code), F.lit(value)).otherwise(class_code)
    mapped = mapped.withColumn("NISS_Class_Codes", class_code)
    class_code = F.col("NISS_Class_Codes")
    for claim_code_value, value in CLAIM_CODE_CLASS_OVERRIDES.items():
        class_code = F.when(F.col("ClaimCode") == claim_code_value, F.lit(value)).otherwise(class_code)
    mapped = (
        mapped.withColumn("NISS_Class_Codes", class_code)
        .withColumn(
            "NISS_Class_Codes",
            F.when(F.col("NISS_Class_Codes") == VEHICLE_CLASS_MARKER, F.col("ClassCode")).otherwise(F.col("NISS_Class_Codes")),
        )
        .withColumn("NISS_Class_Codes", F.substring(F.col("NISS_Class_Codes"), 1, 4))
    )

    corrections = read_lookup(spark, "class_corrections").select(
        F.col("Class").alias("cc_Class"), F.col("NISS Code").alias("cc_NISS_Code")
    )
    return (
        mapped.alias("f")
        .join(corrections.alias("c"), F.col("f.NISS_Class_Codes") == F.col("c.cc_Class"), "left")
        .withColumn(
            "NISS_Class_Codes",
            F.when(F.col("c.cc_NISS_Code").isNotNull(), F.col("c.cc_NISS_Code")).otherwise(F.col("f.NISS_Class_Codes")),
        )
        .drop("cc_Class", "cc_NISS_Code")
    )


def apply_exposure_state_and_fixes(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Derive exposure and state codes, then apply the shared data-fix files.

    Exposure is ProrataPercent x 12, rounded half-up, negative when the premium is negative.
    Origami premium rows with no prorata percentage default to 1.

    Args:
        spark: Active Spark session.
        df: Rows with NISS_Class_Codes.

    Returns:
        The final transformed rows.
    """
    prorata = F.col("ProrataPercent")
    df = df.withColumn(
        "ProrataPercent",
        F.when(
            (F.col("PolicySource") == SOURCE_ORIGAMI) & F.col("ClaimNumber").isNull()
            & (prorata.isNull() | (F.trim(prorata) == "") | (prorata == "null")),
            F.lit(EXPOSURE_DEFAULT_PRORATA),
        ).otherwise(prorata),
    )
    premium = F.when(F.col("OnsetAmount") != 0, F.col("OnsetAmount")).otherwise(F.col("OffsetAmount"))
    raw_exposure = F.col("ProrataPercent") * MONTHS_PER_YEAR
    rounded = F.when((raw_exposure - F.floor(raw_exposure)) >= 0.5, F.ceil(raw_exposure)).otherwise(F.floor(raw_exposure))
    df = df.withColumn("Final_Exposure", F.when(premium < 0, -rounded).otherwise(rounded))

    # State abbreviations become the numeric codes used by the shared data-fix files.
    state = F.col("PrimaryRiskState")
    state_code = state
    for abbreviation, number in reversed(list(STATE_CODE_MAP.items())):
        state_code = F.when(state == abbreviation, F.lit(number)).otherwise(state_code)
    df = df.withColumn("PrimaryRiskState", state_code)

    if CLAIM_COVERAGE_CODE_FIX:
        claim_number, claim_coverage, new_code = CLAIM_COVERAGE_CODE_FIX
        df = df.withColumn(
            "CoverageCode",
            F.when(
                (F.col("ClaimNumber") == claim_number) & (F.col("ClaimCoverageName") == claim_coverage), new_code
            ).otherwise(F.col("CoverageCode")),
        )

    offset_keys = ["InTransactionId", "PolicyNumber", "PolicyVersion", "PrimaryRiskState", "CoverageCode", "OffsetAmount"]
    df = apply_data_fix(df, read_lookup(spark, "datafix_offsets"), offset_keys, claim_rows_only=False)
    claim_keys = ["ClaimNumber", "InTransactionId", "CoverageCode"]
    return apply_data_fix(df, read_lookup(spark, "datafix_claims"), claim_keys, claim_rows_only=True)


def apply_data_fix(df: DataFrame, fix_df: DataFrame, join_columns: List[str], claim_rows_only: bool) -> DataFrame:
    """Overwrite columns with the non-null values of a data-fix file.

    Args:
        df: Rows to correct.
        fix_df: Data-fix file. Columns that are not join columns are the columns to overwrite.
        join_columns: Columns that identify the rows to correct.
        claim_rows_only: When True, only rows with a ClaimNumber are corrected.

    Returns:
        The rows with the corrections applied.
    """
    update_columns = [name for name in fix_df.columns if name not in join_columns]
    joined = df.alias("f2").join(fix_df.alias("f1"), on=join_columns, how="left")

    def corrected(name: str) -> Column:
        """Return the corrected version of one column."""
        fix_value = F.col("f1.%s" % name)
        condition = fix_value.isNotNull()
        if claim_rows_only:
            condition = condition & F.col("f2.ClaimNumber").isNotNull()
        return F.when(condition, fix_value).otherwise(F.col("f2.%s" % name)).alias(name)

    return joined.select(
        *[corrected(name) if name in update_columns else F.col("f2.%s" % name) for name in df.columns]
    )


def build_transformed_table(spark: SparkSession) -> DataFrame:
    """Combine policy and claim rows, apply all attribute rules and save the result.

    Args:
        spark: Active Spark session.

    Returns:
        The saved transformed table as a DataFrame.
    """
    combined = combine_policy_and_claims(spark)
    ruled = apply_extension_group_and_aslob(spark, combined)
    coded = apply_pip_indicator_and_claim_codes(spark, ruled)
    classified = derive_niss_class_code(spark, coded)
    final = apply_exposure_state_and_fixes(spark, classified)
    return save_table(spark, final, il_table("Final_Transformed_Data"))


## 8. Illinois rows, class mapping, aggregation and rules

Everything below reads only the Illinois-prefixed table created above.


In [ ]:
def add_policy_dates(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Attach the transaction effective date and the term expiration date to every row.

    The dates come from policy_policy_detail_info, matched on policy number and policy version.
    An offset row carries its own policy version, so it takes the dates of its own transaction.

    Args:
        spark: Active Spark session.
        df: The Illinois transformed table.

    Returns:
        The rows with ep_txn_eff and ep_term_exp columns.

    Raises:
        ValueError: If a configured date column is missing from policy_policy_detail_info.
    """
    policies = spark.table("%s.policy_policy_detail_info" % POLICY_SCHEMA)
    available = {name.lower() for name in policies.columns}
    missing = [
        name for name in (POLICY_TRANSACTION_EFFECTIVE_COLUMN, POLICY_TERM_EXPIRATION_COLUMN)
        if name.lower() not in available
    ]
    if missing:
        raise ValueError(
            "policy_policy_detail_info has no column %s. Earned premium cannot be calculated; "
            "set the date column parameters." % missing
        )
    dates = (
        policies.select(
            F.col("PolicyNumber").alias("ep_PolicyNumber"),
            F.col("PolicyVersion").cast("int").alias("ep_PolicyVersion"),
            F.col(POLICY_TRANSACTION_EFFECTIVE_COLUMN).alias("ep_txn_eff"),
            F.col(POLICY_TERM_EXPIRATION_COLUMN).alias("ep_term_exp"),
        )
        .dropDuplicates(["ep_PolicyNumber", "ep_PolicyVersion"])
    )
    joined = (
        df.join(
            dates,
            (df["PolicyNumber"] == dates["ep_PolicyNumber"])
            & (df["PolicyVersion"].cast("int") == dates["ep_PolicyVersion"]),
            "left",
        )
        .drop("ep_PolicyNumber", "ep_PolicyVersion")
    )
    no_dates = joined.filter(
        F.col("ClaimNumber").isNull() & (F.col("ep_txn_eff").isNull() | F.col("ep_term_exp").isNull())
    ).count()
    if no_dates:
        logger.warning("%d premium rows have no transaction or expiration date; their earned premium is 0.", no_dates)
    return joined


def standardize_input(df: DataFrame, window: Dict[str, object]) -> DataFrame:
    """Reduce the transformed table to canonical columns and derive the row-level measures.

    Written premium is the onset amount when it is non-zero, otherwise the offset amount, and only
    for premium rows booked in the experience year (or with no book date). Earned premium is that
    premium multiplied by the days of its period that fall in the experience year, divided by the
    days of the whole period. Written exposure comes only from written premium rows of bodily
    injury coverages, so each vehicle counts once.

    Args:
        df: The Illinois transformed table with the ep_txn_eff and ep_term_exp columns.
        window: Output of year_window.

    Returns:
        A DataFrame with the canonical columns used by the remaining steps. Premium rows that are
        neither written nor earned in the experience year are removed.
    """
    def number(name: str) -> Column:
        """Return a column as a double, with nulls as zero."""
        return F.coalesce(F.col(name).cast("double"), F.lit(0.0))

    onset = number("OnsetAmount")
    offset = number("OffsetAmount")
    premium = F.when(onset != 0, onset).otherwise(offset)
    is_claim_row = F.col("ClaimNumber").isNotNull()
    written_flag = (~is_claim_row) & (F.col("BookDate").isNull() | (F.year(F.col("BookDate")) == window["year"]))

    period_start = F.col("ep_txn_eff").cast("date")
    period_end = F.col("ep_term_exp").cast("date")
    year_start = F.lit(window["start"]).cast("date")
    year_end = F.lit(window["end_of_year"]).cast("date")
    total_days = F.datediff(period_end, period_start)
    days_in_year = F.greatest(
        F.lit(0), F.datediff(F.least(period_end, year_end), F.greatest(period_start, year_start))
    )
    earned_premium = F.when(
        is_claim_row | period_start.isNull() | period_end.isNull() | (total_days <= 0), F.lit(0.0)
    ).otherwise(premium * days_in_year / total_days)

    coverage_code = F.upper(F.trim(F.col("CoverageCode").cast("string")))
    bi_codes = [value.strip().upper() for value in BI_COVERAGE_CODES]
    is_bi_row = coverage_code.isin(bi_codes)

    standardized = df.select(
        F.col("PolicyNumber").cast("string").alias("policy_number"),
        F.col("PolicyVersion").cast("string").alias("policy_version"),
        F.col("ClaimNumber").cast("string").alias("claim_number"),
        F.trim(F.col("PrimaryRiskState").cast("string")).alias("state"),
        F.col("ASLOB").cast("double").cast("int").cast("string").alias("aslob"),
        coverage_code.alias("coverage_code"),
        F.trim(F.col(CLASS_SOURCE_COLUMN).cast("string")).alias("class_raw"),
        written_flag.alias("written_flag"),
        F.when(written_flag, premium).otherwise(F.lit(0.0)).alias("written_premium"),
        earned_premium.alias("earned_premium"),
        F.when(written_flag & is_bi_row, number("Final_Exposure")).otherwise(F.lit(0.0)).alias("written_exposure"),
        number("PaidLossAmount").alias("paid_loss"),
        number("ReserveAmount").alias("outstanding_loss"),
        number("AllocatedLossAdjExp").alias("paid_alae"),
        number("PaidCount").alias("paid_claims"),
        number("ReserveCount").alias("outstanding_claims"),
    )
    return standardized.filter(
        F.col("claim_number").isNotNull() | F.col("written_flag") | (F.col("earned_premium") != 0)
    )


def classify_rows(df: DataFrame) -> DataFrame:
    """Label every row as kept or excluded from Line 19.4.

    In the exclude_listed mode (instructions.txt) the rows with an ASLOB in ASLOB_EXCLUDE_CODES are
    removed and rows with no ASLOB are kept and flagged. In the keep_listed mode only the rows
    with an ASLOB in LIABILITY_ASLOB_CODES are kept.

    Args:
        df: Output of standardize_input.

    Returns:
        The DataFrame with a row_status column: KEPT, KEPT_ASLOB_MISSING, EXCLUDED_ASLOB,
        NOT_LIABILITY_ASLOB, ASLOB_MISSING or EXCLUDED_COVERAGE.
    """
    excluded_coverage = [value.strip().upper() for value in EXCLUDED_COVERAGE_CODES]
    aslob = F.col("aslob")
    if ASLOB_FILTER_MODE == "keep_listed":
        status = (
            F.when(aslob.isNull(), F.lit("ASLOB_MISSING"))
            .when(~aslob.isin(LIABILITY_ASLOB_CODES), F.lit("NOT_LIABILITY_ASLOB"))
        )
    else:
        status = F.when(aslob.isin([str(code_value) for code_value in ASLOB_EXCLUDE_CODES]), F.lit("EXCLUDED_ASLOB"))
    status = (
        status.when(F.col("coverage_code").isin(excluded_coverage), F.lit("EXCLUDED_COVERAGE"))
        .when(aslob.isNull(), F.lit("KEPT_ASLOB_MISSING"))
        .otherwise(F.lit(ROW_STATUS_KEPT))
    )
    return df.withColumn("row_status", status)


def log_row_status_summary(df: DataFrame) -> None:
    """Log row counts and totals per row status and per ASLOB value, so exclusions can be reconciled.

    Args:
        df: Output of classify_rows.
    """
    summary = df.groupBy("row_status").agg(
        F.count(F.lit(1)).alias("rows"),
        F.sum("written_premium").alias("written_premium"),
        F.sum("earned_premium").alias("earned_premium"),
        F.sum("paid_loss").alias("paid_loss"),
        F.sum("outstanding_loss").alias("outstanding_loss"),
    )
    for row in summary.collect():
        message = "%-20s rows=%s written_premium=%s earned_premium=%s paid_loss=%s outstanding_loss=%s"
        values = (
            row["row_status"], row["rows"], row["written_premium"], row["earned_premium"],
            row["paid_loss"], row["outstanding_loss"],
        )
        if "ASLOB_MISSING" in row["row_status"]:
            logger.warning(message, *values)
        else:
            logger.info(message, *values)
    by_aslob = df.groupBy("aslob").agg(F.count(F.lit(1)).alias("rows")).collect()
    logger.info("Rows by ASLOB: %s", {row["aslob"]: row["rows"] for row in by_aslob})


def load_class_rules() -> pd.DataFrame:
    """Read the class workbook and return one row per class prefix.

    Each configured sheet gives a list of class code prefixes and the Illinois class they map to.
    Codes are read as text so leading zeros are kept.

    Returns:
        A pandas DataFrame with class_prefix, illinois_class, prefix_length and priority columns.

    Raises:
        RuntimeError: If the workbook cannot be read.
        ValueError: If a sheet is missing a column, has a code of the wrong length or an empty
            cell, maps one prefix to two classes, or uses a class that Illinois does not define.
    """
    frames = []
    for priority, rule in enumerate(CLASS_WORKBOOK_SHEETS):
        try:
            sheet = pd.read_excel(
                CLASS_WORKBOOK_PATH, sheet_name=rule["sheet"], header=rule["header_row"], dtype=str
            )
        except (FileNotFoundError, ImportError, ValueError) as error:
            raise RuntimeError(
                "Cannot read sheet '%s' of %s (%s). Upload the workbook and check CLASS_WORKBOOK_PATH."
                % (rule["sheet"], CLASS_WORKBOOK_PATH, error)
            ) from error
        missing = [name for name in (rule["code_column"], rule["class_column"]) if name not in sheet.columns]
        if missing:
            raise ValueError("Sheet '%s' has no column %s." % (rule["sheet"], missing))
        part = sheet[[rule["code_column"], rule["class_column"]]].dropna(how="all")
        part.columns = ["class_prefix", "illinois_class"]
        if part.isna().any().any():
            raise ValueError("Sheet '%s' has rows with an empty class code or class." % rule["sheet"])
        part["class_prefix"] = part["class_prefix"].str.strip()
        part["illinois_class"] = part["illinois_class"].str.strip().str.upper()
        wrong = part[part["class_prefix"].str.len() != rule["prefix_length"]]
        if not wrong.empty:
            raise ValueError(
                "Sheet '%s' has codes that are not %d characters long: %s"
                % (rule["sheet"], rule["prefix_length"], wrong["class_prefix"].tolist()[:10])
            )
        part["prefix_length"] = rule["prefix_length"]
        part["priority"] = priority
        frames.append(part)
    rules = pd.concat(frames, ignore_index=True)
    targets = rules.groupby(["prefix_length", "class_prefix"])["illinois_class"].nunique()
    if (targets > 1).any():
        raise ValueError("Some class prefixes map to more than one Illinois class.")
    validate_illinois_class_codes(sorted(rules["illinois_class"].unique()))
    return rules.drop_duplicates(["prefix_length", "class_prefix"]).reset_index(drop=True)


def read_class_workbook(spark: SparkSession) -> DataFrame:
    """Return the class rules as a Spark DataFrame.

    Args:
        spark: Active Spark session.

    Returns:
        A DataFrame with class_prefix, illinois_class, prefix_length and priority columns.
    """
    return spark.createDataFrame(load_class_rules())


def add_illinois_class_code(df: DataFrame, rules: DataFrame) -> DataFrame:
    """Attach the Illinois class code using the class workbook; everything else becomes OTHR.

    The sheets are checked in the order of CLASS_WORKBOOK_SHEETS. The first characters of the
    vehicle class code are compared with the prefixes of each sheet. A class that is not found on
    any sheet, and a row with no vehicle class, are reported as OTHR.

    Args:
        df: Rows in scope for Line 19.4 with a class_raw column.
        rules: Output of read_class_workbook.

    Returns:
        The DataFrame with an illinois_class_code column.
    """
    result = df
    class_columns = []
    helper_columns = []
    for rule in CLASS_WORKBOOK_SHEETS:
        length = rule["prefix_length"]
        key, prefix, mapped = "class_key_%d" % length, "rule_prefix_%d" % length, "rule_class_%d" % length
        sheet_rules = (
            rules.filter(F.col("prefix_length") == length)
            .select(F.col("class_prefix").alias(prefix), F.col("illinois_class").alias(mapped))
        )
        result = (
            result.withColumn(key, F.substring(F.col("class_raw"), 1, length))
            .join(F.broadcast(sheet_rules), F.col(key) == F.col(prefix), "left")
        )
        class_columns.append(F.col(mapped))
        helper_columns.extend([key, prefix, mapped])
    return (
        result.withColumn("illinois_class_code", F.coalesce(*class_columns, F.lit(OTHER_CLASS_CODE)))
        .drop(*helper_columns)
    )


def log_class_profile(df: DataFrame) -> None:
    """Log how the vehicle class codes were mapped, so the matching can be verified.

    Logs the length of the class codes, the totals per Illinois class and the vehicle class codes
    with the largest premium that ended up as OTHR.

    Args:
        df: Output of add_illinois_class_code.
    """
    lengths = df.groupBy(F.length("class_raw").alias("length")).agg(F.count(F.lit(1)).alias("rows")).collect()
    logger.info("Vehicle class code length profile: %s", {row["length"]: row["rows"] for row in lengths})
    totals = df.groupBy("illinois_class_code").agg(
        F.count(F.lit(1)).alias("rows"), F.sum("written_premium").alias("written_premium"),
        F.sum("earned_premium").alias("earned_premium"), F.sum("written_exposure").alias("written_exposure"),
    ).collect()
    for row in totals:
        logger.info(
            "Class %-5s rows=%s written_premium=%s earned_premium=%s exposure=%s",
            row["illinois_class_code"], row["rows"], row["written_premium"], row["earned_premium"], row["written_exposure"],
        )
    top_other = (
        df.filter(F.col("illinois_class_code") == F.lit(OTHER_CLASS_CODE))
        .groupBy("class_raw").agg(F.sum(F.abs(F.col("written_premium"))).alias("premium"))
        .orderBy(F.desc("premium")).limit(10).collect()
    )
    logger.info("Largest vehicle class codes reported as OTHR: %s", [(row["class_raw"], row["premium"]) for row in top_other])


def check_exposure_basis(df: DataFrame) -> None:
    """Stop or warn when a class with a miles or receipts exposure base has data.

    Section 4203.50 defines classes 1B, 1C, 3B and 3C with receipts or miles as exposure. The data
    has car-months only, so MILES_RECEIPTS_CLASS_MODE decides what happens.

    Args:
        df: Output of add_illinois_class_code.

    Raises:
        ValueError: If such a class has data and MILES_RECEIPTS_CLASS_MODE is stop.
    """
    flagged = df.filter(F.col("illinois_class_code").isin(list(EXPOSURE_BASE_OTHER))).agg(
        F.collect_set("illinois_class_code").alias("classes"),
        F.sum(F.abs(F.col("written_premium")) + F.abs(F.col("earned_premium")) + F.abs(F.col("written_exposure"))).alias("amount"),
    ).collect()[0]
    if not flagged["amount"]:
        return
    message = (
        "Classes %s have data, but their exposure base is receipts or miles and the data has car-months "
        "only. Set MILES_RECEIPTS_CLASS_MODE to 'car_months' to continue with car-months, or provide "
        "the exposure rule." % sorted(flagged["classes"])
    )
    if MILES_RECEIPTS_CLASS_MODE == "stop":
        raise ValueError(message)
    logger.warning(message)


def build_pre_report(spark: SparkSession, df: DataFrame) -> DataFrame:
    """Save the row-level pre-report detail behind the Illinois report.

    Args:
        spark: Active Spark session.
        df: Output of add_illinois_class_code.

    Returns:
        The saved pre-report table. A CSV copy with a header is written when WRITE_PRE_REPORT_CSV is True.
    """
    detail = df.select(
        "policy_number", "policy_version", "claim_number", "state", "aslob", "coverage_code", "class_raw",
        "illinois_class_code", F.lit(FORM_TYPE).alias("form_type"), "written_flag", *MEASURES,
    )
    saved = save_table(spark, detail, il_table("PreReport_Data"))
    if WRITE_PRE_REPORT_CSV:
        write_single_csv(spark, saved, OUTPUT_FOLDER, PRE_REPORT_FILE_NAME, True)
    return saved


In [ ]:
def claim_level_counts(df: DataFrame) -> DataFrame:
    """Count each claim once per class (Illinois instructions, Section 4203.30i and 4203.30j).

    The STAT row-level counts (1, 0 or -1 per claim coverage row) are added up per claim and class.
    A claim is paid when its net paid indicator is above zero. It is outstanding when its net
    reserve indicator is above zero and it is not paid, so a claim is never counted both ways. A
    claim with rows in several classes counts once in each class.

    Args:
        df: Rows of one scope with illinois_class_code, claim_number, paid_claims and
            outstanding_claims columns.

    Returns:
        One row per class with the paid_claims and outstanding_claims counts.
    """
    per_claim = (
        df.filter(F.col("claim_number").isNotNull())
        .groupBy("illinois_class_code", "claim_number")
        .agg(F.sum("paid_claims").alias("paid_net"), F.sum("outstanding_claims").alias("outstanding_net"))
    )
    paid_indicator = F.signum(F.col("paid_net"))
    outstanding_indicator = F.when(paid_indicator > 0, F.lit(0.0)).otherwise(F.signum(F.col("outstanding_net")))
    return (
        per_claim.withColumn("paid_claims", paid_indicator)
        .withColumn("outstanding_claims", outstanding_indicator)
        .groupBy("illinois_class_code")
        .agg(F.sum("paid_claims").alias("paid_claims"), F.sum("outstanding_claims").alias("outstanding_claims"))
    )


def aggregate_scope(df: DataFrame, scope: str) -> DataFrame:
    """Aggregate the measures per Illinois class for one reporting scope.

    Amounts and exposure are summed. Claim counts follow CLAIM_COUNT_MODE: stat adds up the STAT
    row counts, per_claim counts each claim once per class (see claim_level_counts).

    Args:
        df: Rows in scope with an illinois_class_code column.
        scope: "IL" for Illinois-only or "MS" for Multi-state.

    Returns:
        One row per class with the measures and a state_id column.

    Raises:
        ValueError: If the scope is not "IL" or "MS".
    """
    if scope == "IL":
        scoped = df.filter(F.col("state") == F.lit(STATE_CODE_MAP[ILLINOIS_ABBREVIATION]))
        state_id = ILLINOIS_STATE_ID
    elif scope == "MS":
        scoped = df
        state_id = MULTISTATE_ID
    else:
        raise ValueError("scope must be 'IL' or 'MS'.")
    summed = SUMMED_MEASURES + (COUNT_MEASURES if CLAIM_COUNT_MODE == "stat" else [])
    aggregated = scoped.groupBy("illinois_class_code").agg(*[F.sum(F.col(name)).alias(name) for name in summed])
    if CLAIM_COUNT_MODE == "per_claim":
        aggregated = (
            aggregated.join(claim_level_counts(scoped), on="illinois_class_code", how="left")
            .fillna(0, subset=COUNT_MEASURES)
        )
    return aggregated.withColumn("state_id", F.lit(state_id))


def any_nonzero(columns: List[str]) -> Column:
    """Return a condition that is true when any listed column is non-zero.

    Args:
        columns: Column names to test. Nulls are treated as zero.

    Returns:
        A Spark boolean Column.
    """
    condition = F.lit(False)
    for name in columns:
        condition = condition | (F.coalesce(F.col(name), F.lit(0)) != 0)
    return condition


def round_measures(df: DataFrame) -> DataFrame:
    """Round every measure to a whole number (halves round up, Section 4203.30q).

    Args:
        df: Aggregated DataFrame.

    Returns:
        The DataFrame with measures cast to long.
    """
    for name in MEASURES:
        df = df.withColumn(name, F.round(F.col(name), 0).cast("long"))
    return df


def apply_submission_rules(illinois_df: DataFrame, multistate_df: DataFrame) -> DataFrame:
    """Apply the Illinois record rules and combine both scopes.

    Rules applied:
      * Illinois-only records need premium or loss activity (run-off rule, Section 4203.30k).
      * Multi-state records are kept only for classes that have an Illinois-only record, and
        only when at least one amount field is non-zero.
      * Written exposure is blank for the OTHR class when BLANK_EXPOSURE_FOR_OTHR is True
        (Section 4203.30e).

    Args:
        illinois_df: Output of aggregate_scope for "IL".
        multistate_df: Output of aggregate_scope for "MS".

    Returns:
        The combined, rule-compliant records.
    """
    illinois = round_measures(illinois_df).filter(any_nonzero(ACTIVITY_MEASURES))
    reportable_classes = illinois.select("illinois_class_code").distinct()
    multistate = (
        round_measures(multistate_df)
        .join(reportable_classes, on="illinois_class_code", how="inner")
        .filter(any_nonzero(AMOUNT_MEASURES))
    )
    combined = illinois.unionByName(multistate)
    if not BLANK_EXPOSURE_FOR_OTHR:
        return combined
    return combined.withColumn(
        "written_exposure",
        F.when(F.col("illinois_class_code") == F.lit(OTHER_CLASS_CODE), F.lit(None).cast("long"))
        .otherwise(F.col("written_exposure")),
    )


def is_filing_required(df: DataFrame) -> bool:
    """Return True when Illinois-only data exists for a class other than OTHR.

    Section 4203.30a states that no records are required when OTHR is the only class.

    Args:
        df: Output of apply_submission_rules.

    Returns:
        True if at least one Illinois-only record is for a non-OTHR class.
    """
    classes = {
        row["illinois_class_code"]
        for row in df.filter(F.col("state_id") == F.lit(ILLINOIS_STATE_ID))
        .select("illinois_class_code").distinct().collect()
    }
    return bool(classes - {OTHER_CLASS_CODE})


def assemble_output(df: DataFrame, statistical_year: str) -> DataFrame:
    """Build the 17 output columns in the exact order required by Illinois.

    Every column is written as text. Nulls are written as blanks. Numbers are whole values with a
    leading dash for negatives and no leading zeros.

    Args:
        df: Output of apply_submission_rules.
        statistical_year: Experience year being reported.

    Returns:
        A DataFrame with exactly the columns listed in OUTPUT_COLUMNS.
    """
    def as_text(name: str) -> Column:
        """Return a measure as plain whole-number text."""
        return F.col(name).cast("long").cast("string")

    if OUTSTANDING_ALAE_MODE == "combined":
        outstanding_alae = F.lit(COMBINED_LITERAL)
    elif OUTSTANDING_ALAE_MODE == "combined_when_outstanding":
        outstanding_alae = F.when(
            F.coalesce(F.col("outstanding_loss"), F.lit(0)) != 0, F.lit(COMBINED_LITERAL)
        ).otherwise(F.lit("0"))
    else:
        outstanding_alae = F.lit("0")
    return df.select(
        F.lit(FEIN).alias("FEIN"),
        F.lit(FILING_METHOD).alias("Filing Method"),
        F.lit(LINE_OF_BUSINESS).alias("Line of Business"),
        F.col("state_id").alias("State ID"),
        F.col("illinois_class_code").alias("Class Code"),
        F.lit(statistical_year).alias("Statistical Year"),
        F.lit(None).cast("string").alias("Not Used"),
        F.lit(FORM_TYPE).alias("Form Type"),
        as_text("written_premium").alias("Written Premium"),
        as_text("earned_premium").alias("Earned Premium"),
        as_text("paid_loss").alias("Paid Losses"),
        as_text("outstanding_loss").alias("Outstanding Losses"),
        as_text("paid_alae").alias("Paid ALAE"),
        outstanding_alae.alias("Outstanding ALAE"),
        as_text("written_exposure").alias("Written Exposures"),
        as_text("paid_claims").alias("Number of Paid Claims"),
        as_text("outstanding_claims").alias("Number of Outstanding Claims"),
    ).orderBy("State ID", "Class Code")


def write_single_csv(spark: SparkSession, df: DataFrame, folder: str, file_name: str, header: bool) -> str:
    """Write a DataFrame as one CSV file with a clean name.

    Spark writes a folder of part files. This function writes to a temporary folder, renames the
    single part file to the target name and removes the temporary folder.

    Args:
        spark: Active Spark session.
        df: DataFrame to write.
        folder: Target folder.
        file_name: Target file name.
        header: Whether to write a header row.

    Returns:
        The path of the final file.

    Raises:
        RuntimeError: If Spark did not produce exactly one part file.
    """
    temp_folder = "%s/_tmp_%s" % (folder, file_name)
    target_file = "%s/%s" % (folder, file_name)
    (
        df.coalesce(1).write.format("csv").mode("overwrite")
        .option("header", str(header).lower()).option("nullValue", "").option("emptyValue", "")
        .save(temp_folder)
    )
    hadoop_path = spark._jvm.org.apache.hadoop.fs.Path
    temp_path = hadoop_path(temp_folder)
    file_system = temp_path.getFileSystem(spark._jsc.hadoopConfiguration())
    parts = [
        status.getPath() for status in file_system.listStatus(temp_path)
        if status.getPath().getName().startswith("part-") and status.getPath().getName().endswith(".csv")
    ]
    if len(parts) != 1:
        raise RuntimeError("Expected one part file in %s but found %d." % (temp_folder, len(parts)))
    target_path = hadoop_path(target_file)
    if file_system.exists(target_path):
        file_system.delete(target_path, False)
    file_system.rename(parts[0], target_path)
    file_system.delete(temp_path, True)
    logger.info("Wrote %s", target_file)
    return target_file


def build_output_file_name(statistical_year: str) -> str:
    """Return OUTPUT_FILE_NAME, or a generated name when it is empty.

    Args:
        statistical_year: Experience year being reported.

    Returns:
        The CSV file name.
    """
    if OUTPUT_FILE_NAME:
        return OUTPUT_FILE_NAME
    return "IL_CostContainment_%s_%s.csv" % (LINE_OF_BUSINESS.replace(".", "_"), statistical_year)


def scope_totals(df: DataFrame, book_year: Optional[int]) -> Dict[str, float]:
    """Return premium, loss, ALAE and claim count totals for liability rows of a table.

    Args:
        df: A table with the transformed-table columns (the Illinois table or the STAT table).
        book_year: When set, premium rows are counted only if booked in that year.

    Returns:
        A dictionary of totals.
    """
    def number(name: str) -> Column:
        """Return a column as a double, with nulls as zero."""
        return F.coalesce(F.col(name).cast("double"), F.lit(0.0))

    premium_row = F.col("ClaimNumber").isNull()
    if book_year is not None:
        premium_row = premium_row & (F.year(F.col("BookDate")) == book_year)
    premium = F.when(premium_row, F.when(number("OnsetAmount") != 0, number("OnsetAmount")).otherwise(number("OffsetAmount"))).otherwise(F.lit(0.0))
    aslob = F.col("ASLOB").cast("double").cast("int").cast("string")
    row = df.filter(aslob.isin(LIABILITY_ASLOB_CODES)).agg(
        F.sum(premium).alias("written_premium"),
        F.sum(number("PaidLossAmount")).alias("paid_loss"),
        F.sum(number("ReserveAmount")).alias("reserve"),
        F.sum(number("AllocatedLossAdjExp")).alias("paid_alae"),
        F.sum(number("PaidCount")).alias("paid_claims"),
        F.sum(number("ReserveCount")).alias("reserve_claims"),
    ).collect()[0]
    return {key: float(row[key] or 0.0) for key in row.asDict()}


def reconcile_with_stat(spark: SparkSession, illinois_table: DataFrame, window: Dict[str, object]) -> None:
    """Log Illinois totals next to the STAT table totals (read-only check).

    Premium rows of the Illinois table are counted only when booked in the experience year, as in
    STAT. Differences are expected from the choices that differ from STAT: personal auto classes,
    rows with no book date, the reserve date, the claim-side filters and the ASLOB selection.

    Args:
        spark: Active Spark session.
        illinois_table: The Illinois transformed table.
        window: Output of year_window.
    """
    if not STAT_RECONCILE_TABLE:
        return
    try:
        stat_totals = scope_totals(spark.table(STAT_RECONCILE_TABLE), None)
    except Exception as error:  # noqa: BLE001 - the check is optional.
        logger.warning("STAT reconciliation skipped (%s).", error)
        return
    illinois_totals = scope_totals(illinois_table, window["year"])
    for key in illinois_totals:
        logger.info(
            "Reconcile %-16s Illinois=%.2f STAT=%.2f difference=%.2f",
            key, illinois_totals[key], stat_totals.get(key, 0.0), illinois_totals[key] - stat_totals.get(key, 0.0),
        )


def reconcile_output_totals(classified: DataFrame, records: DataFrame) -> None:
    """Log the in-scope input totals next to the Multi-state record totals.

    Differences are expected only from rounding, from classes dropped by the Illinois rules, and
    from exposure left blank for OTHR.

    Args:
        classified: In-scope rows with an illinois_class_code column.
        records: Output of apply_submission_rules.
    """
    measures = AMOUNT_MEASURES + ["written_exposure"]
    input_totals = classified.agg(*[F.sum(name).alias(name) for name in measures]).collect()[0]
    output_totals = (
        records.filter(F.col("state_id") == F.lit(MULTISTATE_ID))
        .agg(*[F.sum(F.col(name)).alias(name) for name in measures]).collect()[0]
    )
    for name in measures:
        input_value = float(input_totals[name] or 0.0)
        output_value = float(output_totals[name] or 0.0)
        logger.info(
            "Output check %-18s input=%.2f multistate_records=%.2f difference=%.2f",
            name, input_value, output_value, input_value - output_value,
        )


def check_state_page(records: DataFrame) -> None:
    """Compare Illinois-only written premium with the State Page figure, when one is given.

    Args:
        records: Output of apply_submission_rules.
    """
    if STATE_PAGE_WRITTEN_PREMIUM is None:
        return
    row = (
        records.filter(F.col("state_id") == F.lit(ILLINOIS_STATE_ID))
        .agg(F.sum("written_premium").alias("written_premium")).collect()[0]
    )
    premium = float(row["written_premium"] or 0.0)
    if not STATE_PAGE_WRITTEN_PREMIUM:
        logger.warning("State Page written premium is zero; the comparison is skipped.")
        return
    difference = abs(premium - STATE_PAGE_WRITTEN_PREMIUM) / abs(STATE_PAGE_WRITTEN_PREMIUM) * 100.0
    message = "State Page check: Illinois written premium=%.2f State Page=%.2f difference=%.2f percent"
    if difference <= STATE_PAGE_TOLERANCE_PERCENT:
        logger.info(message, premium, STATE_PAGE_WRITTEN_PREMIUM, difference)
    else:
        logger.warning(message + " (above the tolerance)", premium, STATE_PAGE_WRITTEN_PREMIUM, difference)


## 9. Run


In [ ]:
def run() -> Optional[DataFrame]:
    """Run the full pipeline and write the Illinois Line 19.4 CSV.

    Returns:
        The final 17-column DataFrame, or None when no filing is required.
    """
    validate_parameters()
    statistical_year = resolve_statistical_year()
    window = year_window()
    spark = get_spark()

    if OUTSTANDING_ALAE_MODE != "zero":
        logger.warning("Outstanding ALAE is reported as '%s'. Confirm the reserve includes ALAE.", COMBINED_LITERAL)

    # Build the Illinois tables (own names, STAT tables are never touched).
    build_policy_side(spark, window)
    build_claim_side(spark, window)
    transformed = build_transformed_table(spark)
    reconcile_with_stat(spark, transformed, window)

    # Dates for earned premium, row measures, and the ASLOB selection with a log of every exclusion.
    prepared = classify_rows(standardize_input(add_policy_dates(spark, transformed), window))
    log_row_status_summary(prepared)
    in_scope = prepared.filter(F.col("row_status").isin(list(ROW_KEPT_STATUSES))).drop("row_status")

    # Class mapping from the workbook, pre-report detail and class checks.
    classified = add_illinois_class_code(in_scope, read_class_workbook(spark))
    log_class_profile(classified)
    build_pre_report(spark, classified)
    check_exposure_basis(classified)

    # Aggregate both scopes and apply the Illinois rules.
    records = apply_submission_rules(aggregate_scope(classified, "IL"), aggregate_scope(classified, "MS"))
    reconcile_output_totals(classified, records)
    check_state_page(records)
    if not is_filing_required(records):
        logger.warning(
            "Only the %s class is present for Illinois. No records are required; submit an "
            "exemption request instead (Section 4203.30a).", OTHER_CLASS_CODE,
        )
        return None

    # Assemble and write the final file.
    final = assemble_output(records, statistical_year)
    counts = {row["State ID"]: row["count"] for row in final.groupBy("State ID").count().collect()}
    logger.info("Records by State ID: %s", counts)
    write_single_csv(spark, final, OUTPUT_FOLDER, build_output_file_name(statistical_year), WRITE_HEADER)
    return final


report_df = run()
if report_df is not None:
    report_df.show(100, truncate=False)
